# 📘 Tutorial 3: Mixed-Variable and Constrained BO for Realistic Experimental Design Spaces

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 2**, we extended Bayesian Optimisation from single-point sequential decision-making to **batch BO**, where multiple experiments or simulations can be selected together in one optimisation round.

We saw that, even when the surrogate model and acquisition-based logic remain structurally familiar, the practical BO problem changes once the decision unit becomes a **set of points** rather than a single next point.

In particular, we saw that batch BO requires us to think more carefully about:

- how to select multiple points jointly,
- how to compare performance by **BO round** versus **total evaluations**,
- how the internal geometry of a batch affects exploration and exploitation,
- and how the batch itself can sometimes be shaped through explicit scattering or clustering rules.

That established batch BO as a practical workflow for parallel experimental or computational settings.

In this tutorial, we take the next step:

> **what changes when the design space itself is no longer a simple continuous box, but instead contains mixed variable types and explicit feasibility constraints?**

That is the focus of this notebook.

This is an important shift.

In the earlier tutorials, BO was still operating in spaces where the variables were, in essence, all continuous and freely searchable.
Even when the dimensionality increased, and even when the evaluations became batched, the search space itself remained comparatively clean.

That remains a useful baseline.

But in many realistic research problems, the design variables are not all of the same type.

Instead, an optimisation problem may involve:

- continuous variables such as temperature or concentration,
- discrete variables such as reaction time chosen from a small number of allowed levels,
- categorical variables such as solvent or catalyst identity,
- and feasibility rules that make some nominal variable combinations invalid or impractical.

Once that becomes true, the BO problem changes in a meaningful way.

A mixed-variable constrained BO problem is not just:

- the same surrogate,
- plus the same acquisition function,
- plus a few extra if-statements.

It also introduces new practical questions:

- how to represent a design space that is not fully continuous,
- how to convert latent BO proposals into actual experimental settings,
- how to handle categorical and discrete variables within a continuous BO workflow,
- how to enforce feasibility in a practical way,
- and how to decide whether two different latent proposals really correspond to two different experiments.

So this tutorial is not mainly about introducing a completely different optimisation algorithm.

It is about learning how to adapt the BO workflow to a setting where the search space itself has more realistic experimental structure.

To make this concrete, the notebook uses a synthetic **catalyst optimisation** problem with:

- continuous variables,
- discrete variables,
- categorical choices,
- and solvent-dependent feasibility rules.

That lets us study several core questions:

- how to represent a mixed-variable design space in a BO-friendly way,
- how to decode latent BO proposals into user-facing experimental settings,
- how to repair infeasible designs into feasible ones,
- how to define a repaired objective for optimisation,
- and how to compare a **naïve latent-space BO loop** against a **repair-aware BO loop** that explicitly reasons about repaired experimental novelty.

This is also where the notebook connects even more directly to real experimental optimisation.

In realistic research settings, the main challenge is often not just how to choose the next numerically attractive point, but how to choose the next **runnable experiment** in a space where:

- variables are mixed in type,
- constraints matter,
- and many nominal proposals may collapse onto the same feasible condition after repair.

That is exactly the setting this tutorial is designed to approach.

---

**This tutorial is designed to shift perspective**
- from *“I can run BO in a higher-dimensional and batched setting”*
- to *“I can build and interpret BO when the design space itself is mixed-variable, constrained, and experimentally structured.”*

---

**The emphasis is on developing intuition for**
- why mixed-variable BO is not just continuous BO with a few post-processing steps,
- why the meaningful object of optimisation may be the **repaired feasible experiment** rather than the raw latent proposal,
- how decoding and repair change the interpretation of BO candidates,
- why latent-space novelty does not always imply experimental novelty,
- and how a repair-aware BO loop can improve optimisation efficiency by reasoning in terms of feasible repaired designs.

---

**Key ideas explored include**
- constructing a **latent continuous BO representation** for a mixed-variable problem,
- decoding latent proposals into experimental variables,
- repairing infeasible decoded designs into feasible catalyst conditions,
- defining a synthetic repaired objective over those feasible conditions,
- comparing **naïve latent BO** against **repair-aware BO**,
- analysing best-score trajectories in a constrained setting,
- and tracking repaired-design diversity and duplicate repaired-design rejections.

---

This tutorial serves as the bridge from:

- **batch BO in continuous design spaces** in Tutorial 2,
- to **mixed-variable and constrained BO** in more realistic experimental optimisation settings.

In other words:

- **Tutorial 2** showed how to build and interpret BO when one optimisation round can propose multiple points together,
- and **Tutorial 3** now asks what happens when the design space itself contains **continuous, discrete, categorical, and constrained structure** that BO must respect.

---

**Recommended prerequisites**
- Completion of **Tutorials 1–2**
- Familiarity with Gaussian Process surrogates and acquisition functions in BoTorch
- Familiarity with the standard sequential BO loop
- Familiarity with batch BO concepts and best-value trajectory plots
- Basic awareness that realistic experimental design spaces often contain mixed variable types and feasibility constraints

---

**Author**: Angze Li

**Last updated**: 2026-04-12

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch.optim import optimize_acqf
from botorch.utils.sampling import draw_sobol_samples
from botorch.acquisition import LogExpectedImprovement

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

## 0. Defining a mixed-variable constrained catalyst optimisation problem

Before implementing any code, we first define the optimisation problem that this notebook will study.

In earlier tutorials, the BO problems were mainly posed over **continuous box domains**, where every input variable could vary freely within a numerical interval.
That is a very useful starting point, but many realistic optimisation tasks do not look like that.

In real experimental design, it is common for a problem to contain a **mixture of variable types**.

For example, a catalyst optimisation problem may naturally involve:

- a **continuous** variable such as temperature,
- another **continuous** variable such as catalyst loading,
- a **discrete** variable such as reaction time chosen from a small set of allowed levels,
- and a **categorical** variable such as solvent identity.

So mixed-variable design spaces are not unusual or artificial.
They arise naturally whenever an experimental workflow combines:

- tunable operating conditions,
- discrete process settings,
- and choices between qualitatively different materials or environments.

That is exactly the kind of setting considered in this tutorial.

---

### The toy problem studied in this notebook

To make these ideas concrete, we will work with a synthetic **catalyst optimisation** problem with four experimental variables:

- **temperature** — continuous
- **loading** — continuous
- **reaction time** — discrete
- **solvent** — categorical

The aim is to optimise a synthetic performance score that can be interpreted loosely as something like:

- catalytic yield,
- activity,
- selectivity,
- or a general reaction-performance metric

while also respecting a set of simple but meaningful feasibility rules.

This makes the example much closer to a realistic experimental BO scenario than a purely continuous benchmark such as Rosenbrock.

---

### Variable bounds and levels

The user-facing design space is:

- **temperature** in $[40, 140]$
- **loading** in $[0.01, 0.2]$
- **reaction time** in $\{10, 20, 30, 40, 50\}$
- **solvent** in $\{\mathrm{A}, \mathrm{B}, \mathrm{C}\}$

So the problem contains:

- two continuous variables,
- one discrete variable,
- and one categorical variable.

That is what makes it a **mixed-variable** BO problem.

---

### Why mixed variables matter here

This changes the BO problem in an important way.

The design space is no longer just a continuous hyper-rectangle in which every coordinate can be varied smoothly and independently.

Instead:

- some variables must be **rounded or snapped** to allowed values,
- some variables represent **choices between categories** rather than numerical scales,
- and not every nominal combination of variables is actually feasible.

So before BO can evaluate a candidate, the raw proposed point has to be interpreted as an actual experimental condition.

That is why this notebook is built around the pipeline:

> **latent BO candidate → decode → repair → evaluate**

This is the central modelling idea for the tutorial.

---

### Feasibility rules in this toy problem

In addition to the variable bounds and levels, the design space also contains a set of **solvent-dependent feasibility conditions**.

These are deliberately chosen to make the problem structured rather than fully free.

#### Cross-variable compatibility rule
- Temperatures above **110 °C** are only compatible with **solvent C**

#### Solvent A
- temperature must lie between **65 °C and 85 °C**
- loading must be at most **0.12**
- reaction time must belong to **{10, 20}**

#### Solvent B
- temperature must lie between **88 °C and 108 °C**
- loading must lie between **0.08 and 0.13**
- reaction time must belong to **{20, 30}**

#### Solvent C
- temperature must lie between **118 °C and 132 °C**
- loading must lie between **0.06 and 0.10**
- reaction time must belong to **{40, 50}**

So each solvent defines its own feasible operating window.

That means the feasible design space is no longer a single simple box.
It is a structured collection of solvent-specific subregions.

---

### Why these conditions are useful

These rules are synthetic, but they are designed to capture realistic features of experimental optimisation:

- some solvents are only stable in certain temperature windows,
- some operating regimes require longer or shorter times,
- some catalyst loadings are only meaningful within narrower solvent-specific ranges,
- and some high-temperature settings force a change of solvent altogether.

So the purpose of these rules is not just to make the notebook more complicated.
It is to create a more realistic optimisation problem in which:

- variable type matters,
- feasibility matters,
- and different regions of the design space correspond to qualitatively different experimental regimes.

---

### What the BO workflow must handle

Because of this mixed and constrained structure, BO can no longer treat every raw numerical candidate as a directly valid experiment.

Instead, the workflow must:

1. decode the raw latent proposal into experimental variables,
2. repair it if it violates any feasibility rules,
3. and then evaluate the repaired feasible design.

So the optimisation is no longer taking place over a simple continuous space in the usual direct sense.

Instead, BO operates over a **latent proposal space**, while the actual experiments live in the **decoded and repaired experimental space**.

That distinction is one of the main conceptual ideas of the notebook.

---

### Key takeaway

This toy catalyst problem is designed to illustrate how **mixed variables and feasibility constraints arise naturally** in realistic optimisation tasks.

The design space includes:

- continuous variables,
- discrete variables,
- categorical choices,
- and solvent-dependent operating constraints.

So this tutorial is not just about running BO on a more complicated benchmark.

It is about learning how to build BO workflows for the kind of structured design spaces that appear naturally in realistic experimental optimisation.

## 1. Setting up the latent BO space

With the mixed-variable catalyst problem now defined conceptually, we can set up the numerical BO representation that will be used throughout the notebook.

This cell does **not** redefine the problem itself.
Instead, it specifies how that problem will be represented internally for optimisation.

That distinction is important.

The earlier introductory markdown cell described the **experimental design space** in user-facing terms:

- temperature,
- loading,
- reaction time,
- and solvent identity.

This cell now defines the **latent continuous space** in which BO will actually propose candidates.

---

### Why a latent space is introduced

The mixed-variable catalyst problem contains:

- continuous variables,
- a discrete variable,
- and a categorical variable.

That means the experimental design space is not naturally a simple continuous box that can be searched directly with standard BO machinery.

To make BO practical, we therefore introduce a **latent continuous representation**.

The idea is that BO will optimise over a four-dimensional continuous box, and the raw latent candidate will later be transformed through:

> **decode → repair → evaluate**

So this cell is setting up the space in which BO proposes points, not yet the space in which experiments are interpreted.

---

### The random seed

```python
seed = 6
torch.manual_seed(seed)
```

This sets the PyTorch random seed.

The purpose of the seed is reproducibility.

It helps make sure that stochastic components of the notebook — especially:

- the Sobol initial design,
- and later parts of the BO workflow that depend on random initialisation —

behave consistently across reruns.

The choice of `6` does not have any special meaning by itself.
It is simply a fixed seed chosen to make the notebook reproducible.

---

### User-facing variable ranges

The next lines define the ranges and levels of the experimental variables:

```python
TEMP_MIN, TEMP_MAX = 40.0, 140.0
LOAD_MIN, LOAD_MAX = 0.01, 0.20
TIME_LEVELS = np.array([10, 20, 30, 40, 50], dtype=float)
SOLVENTS = ["A", "B", "C"]
```

These are the same user-facing variables introduced conceptually in the earlier markdown cell.

They are included here because the later decoding and repair functions will need explicit numerical bounds and levels.

So this cell provides the numerical constants needed for:

- decoding latent proposals into experimental variables,
- snapping time to allowed discrete levels,
- and enforcing solvent-dependent constraints later on.

---

### The latent BO dimension

The code then defines:

```python
latent_dim = 4
```

This means BO will operate in a **4-dimensional continuous latent space**.

The four latent coordinates are interpreted as:

- `x0` → raw temperature coordinate
- `x1` → raw loading coordinate
- `x2` → raw time coordinate
- `x3` → raw solvent coordinate

So each BO proposal is initially just a point in:

```python
[0, 1]^4
```

This is a convenient internal representation because it allows us to use standard continuous-domain BO tools even though the actual design problem is mixed-variable.

---

### Why all latent coordinates are placed in `[0, 1]`

The latent bounds are defined as:

```python
latent_bounds = torch.tensor(
    [[0.0] * latent_dim, [1.0] * latent_dim],
    dtype=torch.double,
)
```

So the latent search domain is the unit hypercube:

```python
[0, 1]^4
```

This is a very natural choice for BoTorch workflows.

Using a unit box:

- keeps the internal optimisation scale clean,
- makes the latent variables numerically comparable,
- and simplifies later decoding into experimental units

So rather than making BO search directly over temperatures, loadings, time levels, and solvents, it searches over a standardised continuous box and lets the decode step convert those raw values into actual experimental settings.

---

### Why this is useful for mixed-variable BO

This latent-space approach is especially useful because standard BO methods are most naturally formulated over continuous numerical inputs.

By using a latent unit box, we can still:

- fit a Gaussian Process surrogate,
- optimise acquisition functions in the usual way,
- and build a standard BO loop

while leaving the mixed-variable logic to the later decode-and-repair stages.

So this cell is the point where the notebook commits to the modelling strategy:

> **optimise in a continuous latent space, then map proposals into the constrained mixed-variable experimental space**

That is the key practical idea.

---

### The optimisation budget

The cell also defines:

```python
n_init = 8
n_steps = 20
```

These control the BO budget.

#### `n_init = 8`
This means the optimisation will begin with **8 initial Sobol points** in the latent space.

Those points will later be decoded, repaired if necessary, and evaluated.

#### `n_steps = 20`
This means the BO loop will then perform **20 sequential optimisation steps** after the initial design.

So the total number of evaluations at the end of the notebook will be:

```python
8 + 20 = 28
```

assuming the BO loop runs to completion.

This gives a moderate evaluation budget: large enough to show meaningful BO behaviour, but still small enough that the optimisation remains budget-limited in a realistic way.

---

### Why the variable-name lists are useful

The code also defines:

```python
latent_variable_names = ["temp_raw", "loading_raw", "time_raw", "solvent_raw"]
experimental_variable_names = ["temperature", "loading", "time", "solvent"]
```

These names are not mathematically necessary, but they are very useful for interpretation.

They help keep the distinction clear between:

- **latent raw variables**, which BO manipulates directly,
- and **experimental variables**, which are the decoded real-world quantities we care about

This distinction becomes especially important later when we inspect:

- decoded candidates,
- repaired candidates,
- and the best found experimental design

So these names support readability and interpretation throughout the notebook.

---

### What the printed output confirms

The printed lines confirm the key numerical setup:

- latent dimension = 4
- initial design size = 8
- BO steps = 20
- latent bounds shape = `(2, 4)`
- experimental variable names are correctly specified

So this output is simply a quick sanity check that the internal optimisation setup is defined as intended before moving on to decoding and repair.

---

### Key takeaway

This cell sets up the **internal BO representation** of the mixed-variable catalyst problem.

It defines:

- a reproducible random seed,
- the user-facing experimental bounds and levels,
- a 4-dimensional latent unit-box search space,
- and the BO budget through `n_init` and `n_steps`

So while the earlier markdown cell defined the problem conceptually, this cell defines how that problem will actually be represented numerically inside the BO workflow.

In [ ]:
seed = 6
torch.manual_seed(seed)

# User-facing experimental design space
TEMP_MIN, TEMP_MAX = 40.0, 140.0
LOAD_MIN, LOAD_MAX = 0.01, 0.20
TIME_LEVELS = np.array([10, 20, 30, 40, 50], dtype=float)
SOLVENTS = ["A", "B", "C"]

# Latent BO space:
# x0 -> temperature_raw in [0,1]
# x1 -> loading_raw in [0,1]
# x2 -> time_raw in [0,1]
# x3 -> solvent_raw in [0,1]
latent_dim = 4
latent_bounds = torch.tensor(
    [[0.0] * latent_dim, [1.0] * latent_dim],
    dtype=torch.double,
)

n_init = 8
n_steps = 20

latent_variable_names = ["temp_raw", "loading_raw", "time_raw", "solvent_raw"]
experimental_variable_names = ["temperature", "loading", "time", "solvent"]

print("Latent dimension:", latent_dim)
print("Initial design size:", n_init)
print("BO steps:", n_steps)
print("Latent bounds shape:", tuple(latent_bounds.shape))
print("Experimental variables:", experimental_variable_names)

## 2. Decoding latent BO candidates into experimental designs

Now that the latent BO space has been defined, the next step is to explain how a raw BO proposal is turned into an actual experimental setting.

This is the role of **decoding**.

It is a necessary step because BO is not operating directly on the user-facing catalyst design space.
Instead, it is proposing points in the continuous latent unit box:

$$
[0,1]^4
$$

So a raw BO candidate initially looks like something like:

```python
[0.73, 0.18, 0.64, 0.41]
```

but this is **not yet** a meaningful catalyst experiment.

It still has to be interpreted as:

- a temperature,
- a catalyst loading,
- a reaction time,
- and a solvent identity.

That is exactly what decoding does.

---

### Why decoding is needed

The catalyst problem in this notebook contains **mixed variable types**:

- temperature is continuous,
- loading is continuous,
- time is discrete,
- solvent is categorical.

Standard BO tools, however, are most naturally formulated over continuous numerical spaces.

So rather than optimising directly over a mixed-variable design space, we instead let BO search over a continuous latent space and then convert the raw proposal into a valid experimental description afterward.

This gives a practical modelling pipeline:

> **latent candidate → decode → repair → evaluate**

So decoding is the stage that maps a purely numerical BO proposal into something that can be understood as an experimental condition.

---

### What each decoding function does

This cell defines one decoding function for each experimental variable.

#### `decode_temperature(raw_t)`

```python
def decode_temperature(raw_t):
    raw_t = float(np.clip(raw_t, 0.0, 1.0))
    return TEMP_MIN + raw_t * (TEMP_MAX - TEMP_MIN)
```

This takes a latent scalar in `[0,1]` and linearly maps it to the temperature range:

$$
[40,140]
$$

So:

- `0.0` corresponds to `40 °C`
- `1.0` corresponds to `140 °C`
- intermediate values are scaled continuously in between

This is appropriate because temperature is a **continuous variable**.

---

#### `decode_loading(raw_l)`

```python
def decode_loading(raw_l):
    raw_l = float(np.clip(raw_l, 0.0, 1.0))
    return LOAD_MIN + raw_l * (LOAD_MAX - LOAD_MIN)
```

This performs the same type of linear mapping for catalyst loading, converting the latent scalar into the physical loading interval:

$$
[0.01,0.2]
$$

Again, this is a continuous variable, so a smooth linear scaling is natural.

---

#### `decode_time(raw_time)`

```python
def decode_time(raw_time):
    raw_time = float(np.clip(raw_time, 0.0, 1.0))
    continuous_time = TIME_LEVELS[0] + raw_time * (TIME_LEVELS[-1] - TIME_LEVELS[0])
    idx = int(np.argmin(np.abs(TIME_LEVELS - continuous_time)))
    return int(TIME_LEVELS[idx])
```

This is different from the previous two.

Reaction time is **not** allowed to vary continuously.
It must be chosen from the discrete set:

$$
\{ 10, 20, 30, 40, 50\}
$$

So the code first maps the latent scalar into a continuous value between 10 and 50, then snaps it to the **nearest allowed discrete level**.

This is an important example of why decoding is needed in mixed-variable BO:
the latent proposal is continuous, but the experimental variable is discrete.

So decoding performs that translation explicitly.

---

#### `decode_solvent(raw_solvent)`

```python
def decode_solvent(raw_solvent):
    raw_solvent = float(np.clip(raw_solvent, 0.0, 1.0))
    if raw_solvent < 1.0 / 3.0:
        return "A"
    if raw_solvent < 2.0 / 3.0:
        return "B"
    return "C"
```

This handles the **categorical** variable.

Solvent is not numerical in the experimental problem, but BO still proposes a scalar value in `[0,1]`.

So we partition that interval into three bins:

- lower third → solvent A
- middle third → solvent B
- upper third → solvent C

This is a simple but effective way to encode a categorical choice inside a continuous latent BO framework.

So this is the key example of how a continuous latent coordinate can represent a categorical variable through decoding.

---

### Why clipping to `[0,1]` is used

Each decoding function begins with something like:

```python
np.clip(raw_value, 0.0, 1.0)
```

This ensures numerical safety.

Although the BO bounds are already defined over `[0,1]^4`, it is still good practice to clip the value before decoding.
This prevents small numerical overshoots from causing unintended decoded values outside the expected range.

So clipping is simply a robust implementation detail.

---

### How a full experimental design is decoded

The function

```python
def decode_single_design(x_raw):
```

takes one latent candidate vector and applies all four decoding steps together.

It returns a dictionary of the form:

```python
{
    "temperature": ...,
    "loading": ...,
    "time": ...,
    "solvent": ...,
}
```

So this is the first point in the notebook where a raw BO tensor becomes something that actually looks like an experimental condition.

That is important conceptually.

A latent point like:

```python
[0.73, 0.18, 0.64, 0.41]
```

is not very meaningful by itself.
But after decoding, it may become something like:

- temperature = 113.0
- loading = 0.044
- time = 40
- solvent = B

That is a candidate design we can reason about experimentally.

---

### Why `decode_batch(X_raw)` is useful

The final function

```python
def decode_batch(X_raw):
    return [decode_single_design(x_raw) for x_raw in X_raw]
```

simply applies `decode_single_design(...)` to every point in a batch.

This is useful because later parts of the notebook will often work with:

- initial datasets,
- BO histories,
- or candidate collections

rather than just one point at a time.

So `decode_batch(...)` is a convenience function that makes it easy to interpret a whole set of latent BO candidates as a set of experimental designs.

---

### What decoding does *not* do yet

It is important to note that decoding alone does **not** enforce feasibility.

A decoded design may still violate the problem constraints.

For example:

- a decoded solvent A design might have too high a temperature,
- or a solvent C design might decode to an invalid time level for that solvent’s feasible window.

So decoding is only the first step.

It transforms:

- raw latent coordinates
into
- interpretable experimental variables

But the resulting design may still require **repair** before it can be evaluated.

That is why decoding is followed later by the repair stage.

---

### Key takeaway

This cell defines how a raw latent BO proposal is translated into a user-facing catalyst design.

We need decoding because BO is searching over a continuous latent box, while the actual optimisation problem contains:

- continuous variables,
- a discrete variable,
- and a categorical variable.

The decoding functions achieve this by:

- linearly scaling latent values for continuous variables,
- snapping latent values to allowed discrete levels for time,
- and binning a latent scalar into solvent categories.

So decoding is the mechanism that makes standard continuous-domain BO usable in a mixed-variable optimisation problem.

In [ ]:
def decode_temperature(raw_t):
    raw_t = float(np.clip(raw_t, 0.0, 1.0))
    return TEMP_MIN + raw_t * (TEMP_MAX - TEMP_MIN)

def decode_loading(raw_l):
    raw_l = float(np.clip(raw_l, 0.0, 1.0))
    return LOAD_MIN + raw_l * (LOAD_MAX - LOAD_MIN)

def decode_time(raw_time):
    raw_time = float(np.clip(raw_time, 0.0, 1.0))
    continuous_time = TIME_LEVELS[0] + raw_time * (TIME_LEVELS[-1] - TIME_LEVELS[0])
    idx = int(np.argmin(np.abs(TIME_LEVELS - continuous_time)))
    return int(TIME_LEVELS[idx])

def decode_solvent(raw_solvent):
    raw_solvent = float(np.clip(raw_solvent, 0.0, 1.0))
    if raw_solvent < 1.0 / 3.0:
        return "A"
    if raw_solvent < 2.0 / 3.0:
        return "B"
    return "C"

def decode_single_design(x_raw):
    x = x_raw.detach().cpu().numpy().reshape(-1)
    return {
        "temperature": decode_temperature(x[0]),
        "loading": decode_loading(x[1]),
        "time": decode_time(x[2]),
        "solvent": decode_solvent(x[3]),
    }

def decode_batch(X_raw):
    return [decode_single_design(x_raw) for x_raw in X_raw]

## 3. Repairing decoded designs into feasible experimental settings

Decoding converts a latent BO proposal into an interpretable catalyst design, but that decoded design is **not yet guaranteed to be feasible**.

This cell introduces the second stage of the workflow:

> **latent candidate → decode → repair → evaluate**

The purpose of repair is to ensure that every candidate eventually evaluated by the objective corresponds to a **valid experimental condition** under the rules of the toy catalyst problem.

This is a central idea in the notebook.

In a mixed-variable constrained problem, a decoded candidate may violate one or more solvent-dependent feasibility rules.
Rather than rejecting such a design outright, we instead **repair** it by mapping it back into the feasible region.

That makes the optimisation workflow much more realistic for experimental design, where invalid or incompatible settings would often be adjusted into the nearest runnable condition rather than simply discarded.

---

### What the repair stage is doing conceptually

The decoded design now has user-facing variables such as:

- temperature,
- loading,
- time,
- and solvent.

But those values may not be compatible with the feasibility rules introduced earlier.

So the repair stage acts as a **projection-like step** from the decoded design space into the feasible experimental space.

This means:

- infeasible temperatures are clipped into solvent-specific windows,
- invalid times are snapped to solvent-allowed levels,
- incompatible solvents may be changed,
- and continuous variables may be snapped to experimentally realistic grids.

So this stage is where the structured design-space constraints become operational.

---

### `nearest_allowed_time(...)`

```python
def nearest_allowed_time(time_value, allowed_times):
    ...
```

This helper function takes a decoded time and snaps it to the nearest allowed time level from a given subset.

This is useful because different solvents are only compatible with different subsets of the global time levels.

For example:

- solvent A only allows `{10, 20}`
- solvent B only allows `{20, 30}`
- solvent C only allows `{40, 50}`

So this helper is a convenient way to enforce solvent-specific time windows.

---

### `snap_to_grid(...)`

```python
def snap_to_grid(value, step, lower=None, upper=None):
    ...
```

This helper snaps a continuous value onto a fixed numerical grid.

It is used here to make repaired experimental settings look more realistic.

For example:

- temperature is snapped to the nearest **5 °C**
- loading is snapped to the nearest **0.01**

This reflects the fact that real experimental variables are often not controlled with arbitrary continuous precision.

So even after constraint repair, the final design is still expressed in a way that resembles a practical experimental setting.

---

### `repair_single_design(...)`

This is the core function of the cell.

It takes one decoded catalyst design and applies the full feasibility logic.

The repair process is rule-based and solvent-dependent.

---

### Step 1: cross-variable compatibility rule

The first rule is:

```python
if repaired["temperature"] > 110.0 and repaired["solvent"] != "C":
    repaired["solvent"] = "C"
```

This means that very high-temperature conditions are only compatible with solvent C.

So if a decoded design proposes:

- a temperature above 110 °C
- together with solvent A or B

then the solvent is repaired to C.

This is important because it introduces a genuinely mixed-variable interaction:
the validity of the solvent depends on the temperature.

That is much more realistic than treating each variable independently.

---

### Step 2: solvent-specific feasible windows

After that, the function applies solvent-specific rules.

#### Solvent A
For solvent A:

- temperature must remain in `[65, 85]`
- loading must be at most `0.12`
- time must belong to `{10, 20}`

So if the decoded design violates any of these, it is repaired back into that feasible window.

#### Solvent B
For solvent B:

- temperature must remain in `[88, 108]`
- loading must remain in `[0.08, 0.13]`
- time must belong to `{20, 30}`

Again, any violation is repaired back into the valid region.

#### Solvent C
For solvent C:

- temperature must remain in `[118, 132]`
- loading must remain in `[0.06, 0.10]`
- time must belong to `{40, 50}`

So solvent C occupies a higher-temperature, narrower operating regime.

These solvent-specific repairs make the feasible design space a structured collection of subregions rather than one continuous box.

---

### Step 3: snapping to realistic experimental grids

After solvent-specific clipping and time correction, the function then snaps the repaired continuous variables to practical experimental grids:

```python
snapped_temperature = snap_to_grid(..., step=5.0, ...)
snapped_loading = snap_to_grid(..., step=0.01, ...)
```

This means that even a repaired continuous value is not left as an arbitrary real number.

Instead, the final repaired design is expressed in experimentally plausible increments.

This is a useful modelling choice because it makes the toy problem feel more like something that could actually be run in a lab.

It also increases the chance that many different raw latent proposals collapse onto the same repaired experimental setting, which is important later when comparing naïve BO and repair-aware BO.

---

### Repair flags and metadata

The function also records what kinds of repairs were applied.

It stores:

- `was_repaired`
- `repair_flags`
- `repair_count`

This is very useful later because it allows us to analyse:

- how often BO proposals were infeasible,
- what kinds of repairs were most common,
- and how different optimisation strategies interact with the repair process.

So the repair function is not only transforming candidates, but also producing useful diagnostic information.

---

### `repair_batch(...)`

```python
def repair_batch(decoded_batch):
    return [repair_single_design(d) for d in decoded_batch]
```

This simply applies the repair function to every decoded design in a collection.

As with decoding, this is a convenience function that becomes useful whenever the notebook works with batches or datasets rather than single candidates.

---

### `repaired_design_key(...)`

```python
def repaired_design_key(repaired_design):
    return (
        int(round(float(repaired_design["temperature"]))),
        round(float(repaired_design["loading"]), 2),
        int(repaired_design["time"]),
        repaired_design["solvent"],
    )
```

This function converts a repaired design into a compact hashable key.

It is especially important later for the **repair-aware BO loop**, where we want to detect whether a newly proposed latent candidate would repair to a feasible design that has already been seen before.

This is one of the most important practical consequences of repair:

> many different raw latent points may collapse onto the same repaired feasible experiment.

The repaired-design key is what lets the notebook track that phenomenon explicitly.

---

### Why this repair step matters

This cell is doing much more than simple clipping.

It is establishing the whole constrained mixed-variable logic of the notebook.

Without this repair stage:

- decoded designs could remain infeasible,
- the objective would not correspond to a runnable experimental setting,
- and the later comparison between naïve BO and repair-aware BO would not make sense.

So this is one of the most important cells in the tutorial.

It defines the actual feasible experimental space in which the objective will be evaluated.

---

### Key takeaway

This cell introduces the **repair** stage of the mixed-variable BO workflow.

It takes decoded experimental designs and maps them into solvent-dependent feasible operating windows by:

- enforcing a cross-variable solvent–temperature compatibility rule,
- clipping temperature and loading into solvent-specific ranges,
- snapping time to solvent-allowed discrete levels,
- and snapping continuous variables to realistic experimental grids.

So while decoding makes a latent BO proposal interpretable, repair is what makes it **experimentally feasible**.

In [ ]:
def nearest_allowed_time(time_value, allowed_times):
    allowed_times = np.array(sorted(allowed_times), dtype=int)
    idx = int(np.argmin(np.abs(allowed_times - int(time_value))))
    return int(allowed_times[idx])


def snap_to_grid(value, step, lower=None, upper=None):
    snapped = step * round(value / step)
    if lower is not None:
        snapped = max(lower, snapped)
    if upper is not None:
        snapped = min(upper, snapped)
    return float(snapped)


def repair_single_design(decoded_design):
    repaired = dict(decoded_design)
    repair_flags = []

    # Cross-variable rule: temperatures above 110 C are only compatible with solvent C.
    if repaired["temperature"] > 110.0 and repaired["solvent"] != "C":
        repaired["solvent"] = "C"
        repair_flags.append("high_temp_requires_C")

    # Narrow solvent-specific feasible windows.
    if repaired["solvent"] == "A":
        if repaired["temperature"] < 65.0:
            repaired["temperature"] = 65.0
            repair_flags.append("A_temp_floor")
        if repaired["temperature"] > 85.0:
            repaired["temperature"] = 85.0
            repair_flags.append("A_temp_cap")
        if repaired["loading"] > 0.12:
            repaired["loading"] = 0.12
            repair_flags.append("A_loading_cap")

        repaired_time = nearest_allowed_time(repaired["time"], [10, 20])
        if repaired_time != repaired["time"]:
            repaired["time"] = repaired_time
            repair_flags.append("A_time_window")

    elif repaired["solvent"] == "B":
        if repaired["temperature"] < 88.0:
            repaired["temperature"] = 88.0
            repair_flags.append("B_temp_floor")
        if repaired["temperature"] > 108.0:
            repaired["temperature"] = 108.0
            repair_flags.append("B_temp_cap")
        if repaired["loading"] > 0.13:
            repaired["loading"] = 0.13
            repair_flags.append("B_loading_cap")
        if repaired["loading"] < 0.08:
            repaired["loading"] = 0.08
            repair_flags.append("B_loading_floor")

        repaired_time = nearest_allowed_time(repaired["time"], [20, 30])
        if repaired_time != repaired["time"]:
            repaired["time"] = repaired_time
            repair_flags.append("B_time_window")

    elif repaired["solvent"] == "C":
        if repaired["temperature"] < 118.0:
            repaired["temperature"] = 118.0
            repair_flags.append("C_temp_floor")
        if repaired["temperature"] > 132.0:
            repaired["temperature"] = 132.0
            repair_flags.append("C_temp_cap")
        if repaired["loading"] < 0.06:
            repaired["loading"] = 0.06
            repair_flags.append("C_loading_floor")
        if repaired["loading"] > 0.10:
            repaired["loading"] = 0.10
            repair_flags.append("C_loading_cap")

        repaired_time = nearest_allowed_time(repaired["time"], [40, 50])
        if repaired_time != repaired["time"]:
            repaired["time"] = repaired_time
            repair_flags.append("C_time_window")

    # Snap repaired continuous variables to an experimental grid.
    snapped_temperature = snap_to_grid(repaired["temperature"], step=5.0, lower=TEMP_MIN, upper=TEMP_MAX)
    if snapped_temperature != repaired["temperature"]:
        repaired["temperature"] = snapped_temperature
        repair_flags.append("temp_grid_snap")

    snapped_loading = snap_to_grid(repaired["loading"], step=0.01, lower=LOAD_MIN, upper=LOAD_MAX)
    if snapped_loading != repaired["loading"]:
        repaired["loading"] = snapped_loading
        repair_flags.append("loading_grid_snap")

    repaired["was_repaired"] = len(repair_flags) > 0
    repaired["repair_flags"] = repair_flags
    repaired["repair_count"] = len(repair_flags)
    return repaired


def repair_batch(decoded_batch):
    return [repair_single_design(d) for d in decoded_batch]


def repaired_design_key(repaired_design):
    return (
        int(round(float(repaired_design["temperature"]))),
        round(float(repaired_design["loading"]), 2),
        int(repaired_design["time"]),
        repaired_design["solvent"],
    )

## 4. Defining the synthetic catalyst score and the evaluated objective

Once the mixed-variable design space and repair rules have been defined, we still need one final ingredient before BO can begin:

> a scalar objective that tells us how good a repaired catalyst design actually is.

That is the role of the **score** introduced in this cell.

---

### Why a score is needed

Bayesian Optimisation is ultimately an optimisation method.
So even if we can:

- represent the design space,
- decode latent proposals,
- and repair infeasible settings,

BO still needs a numerical target to optimise.

In a real catalyst optimisation problem, that target might be something like:

- yield,
- conversion,
- selectivity,
- stability,
- turnover frequency,
- or some weighted combination of several performance metrics.

In this notebook, we do not have real experimental measurements.
So instead, we construct a **synthetic score** that plays the same role.

This lets the notebook study:

- mixed variables,
- categorical effects,
- feasibility constraints,
- and repair-aware BO behaviour,

without needing actual laboratory data.

So the score is the stand-in for a real experimental performance measurement.

---

### Why the score is introduced only after decoding and repair

A crucial point is that the score is **not** defined directly on the raw latent BO coordinates.

It is defined on the **repaired experimental design**.

That is important because the real object of interest is not:

- an arbitrary latent point in $[0,1]^4$,

but rather:

- the catalyst condition that would actually be run in practice after decoding and repair.

So the evaluation pipeline becomes:

> **raw latent proposal → decode → repair → score**

This means the score represents the quality of the **feasible experiment**, not merely the quality of a raw BO proposal.

That is exactly the modelling choice we want in a constrained experimental BO workflow.

---

### Why a synthetic score is useful in this tutorial

The score is synthetic, but it is designed to have several useful properties.

It should:

- depend on all four experimental variables,
- behave differently for different solvents,
- make some regions genuinely better than others,
- and create a meaningful optimisation landscape for BO to search over.

So rather than making the tutorial purely about bookkeeping and constraints, the score makes the problem a real optimisation task.

Without a score, the notebook would only define a feasible design space.
With the score, it becomes a **mixed-variable constrained optimisation problem**.

---

### Solvent-specific behaviour

The dictionary `SOLVENT_PARAMS` defines a separate set of parameters for each solvent:

- `A`
- `B`
- `C`

Each solvent has its own:

- baseline level (`base`)
- peak amplitude (`amp`)
- preferred temperature (`T_opt`)
- preferred loading (`L_opt`)
- characteristic widths (`sigma_T`, `sigma_L`)
- preferred time (`time_opt`)
- time penalty scale
- and solvent-specific bonus

This is important because it means solvent is not just a cosmetic label.
Changing the solvent changes the entire local structure of the score landscape.

So the categorical variable has a genuine and meaningful effect on optimisation behaviour.

---

### Continuous peak term

The main term in the score is:

```python
continuous_peak = p["amp"] * np.exp(
    -0.5 * ((T - p["T_opt"]) / p["sigma_T"]) ** 2
    -0.5 * ((L - p["L_opt"]) / p["sigma_L"]) ** 2
)
```

This creates a solvent-specific Gaussian-like peak in the two continuous variables:

- temperature
- loading

So each solvent has a preferred region in $begin:math:text$\(T\, L\)$end:math:text$-space where the score is highest.

This makes the objective easier to interpret than something completely arbitrary or highly oscillatory.

It gives each solvent a physically plausible “good operating window.”

---

### Time term

The score also includes:

```python
time_term = -p["time_penalty"] * (t - p["time_opt"]) ** 2
```

This means each solvent has a preferred reaction time, and moving away from that preferred time reduces the score.

That is useful because reaction time is a **discrete variable** in this problem, so this term ensures that time has a genuine effect rather than being a passive label.

So the objective now meaningfully depends on all variable types:

- continuous,
- discrete,
- and categorical.

---

### Interaction term

The term

```python
interaction_term = 0.8 * np.sin(T / 9.0) * np.cos(35.0 * L)
```

adds a small nontrivial interaction between temperature and loading.

This is useful because it prevents the objective from being a completely smooth, perfectly symmetric peak.
It adds a little local structure and mild irregularity, making the optimisation problem more realistic.

So the score is not just a simple Gaussian hill.
It has enough complexity to behave more like a genuine black-box objective.

---

### Edge penalties

The `edge_penalty` block deliberately makes some common repaired-edge regions mediocre.

For example:

- solvent A at `T = 85`
- solvent B at `T = 88`, `T = 108`, or `L = 0.13`
- solvent C at `T = 118`, `T = 132`, `L = 0.06`, or `L = 0.10`

receive penalties.

This is a deliberate modelling choice.

Why?

Because many repaired designs tend to accumulate at solvent-specific boundaries.
If those repaired-edge regions were always highly competitive, then naïve BO and repair-aware BO could look too similar.

By making common repaired edges somewhat mediocre, the notebook creates a stronger incentive for BO to find genuinely good interior feasible regions instead of repeatedly collapsing onto obvious repaired boundaries.

This makes the later comparison between naïve and repair-aware BO much more informative.

---

### Ridge term

The `ridge_term` introduces a mild solvent-specific directional bias in the score landscape.

This helps make each solvent region feel a little more structured and less perfectly symmetric.
So the synthetic objective becomes richer than a simple independent peak in temperature, loading, and time.

Again, the point is not to make the function overly complicated, but to make it plausible enough that BO behaviour is interesting.

---

### The final score

The final score is:

```python
score =
    p["base"] + p["bonus"] +
    continuous_peak +
    time_term + interaction_term + ridge_term +
    edge_penalty
```

So the score combines:

- a solvent-specific baseline,
- a main peak in continuous space,
- a discrete time preference,
- a mild interaction term,
- and a penalty for common repaired-edge regions.

This creates a synthetic but meaningful catalyst-performance landscape.

---

### `evaluate_repaired_design(...)`

This function takes one repaired catalyst design and computes its score.

This is the “true” synthetic experimental response of the notebook.

So whenever we talk about a candidate’s performance, this is the function that determines it.

---

### `objective_mixed_raw(...)`

This is the black-box objective actually used by BO.

It takes a raw latent tensor `X_raw`, and for each point it:

1. decodes it,
2. repairs it,
3. evaluates the repaired design,
4. and returns the resulting score tensor.

This is one of the most important functions in the notebook, because it wraps the whole modelling pipeline together into a form that BO can optimise.

So from the BO algorithm’s perspective, this is the objective.

But conceptually, it is not a direct function of raw latent space.
It is a latent-space wrapper around a repaired experimental score.

---

### Why `batch_to_dataframe(...)` is included here

The helper `batch_to_dataframe(...)` is used to inspect a batch of raw latent candidates together with:

- decoded values,
- repaired values,
- repair flags,
- and optionally the score.

This is extremely useful for interpretation.

Without it, the notebook would mostly manipulate tensors that are hard to understand experimentally.
With this dataframe helper, we can directly inspect what BO is proposing in user-facing catalyst terms.

So this function makes the mixed-variable and constrained structure much easier to interpret.

---

### Key takeaway

This cell introduces the **synthetic catalyst score**, which is the objective BO will optimise after decoding and repair.

We need this score because BO ultimately requires a scalar target that measures how good a feasible design is.

In this notebook, the score serves as a stand-in for a real catalyst-performance metric such as yield or activity.

It is defined on the **repaired experimental design**, not the raw latent coordinates, so that the optimisation always reflects the quality of a valid runnable experiment.

That is what turns the mixed-variable constrained design space into a genuine Bayesian Optimisation problem.

In [ ]:
SOLVENT_PARAMS = {
    "A": {
        "base": 40.0,
        "amp": 22.0,
        "T_opt": 75.0,
        "L_opt": 0.09,
        "sigma_T": 5.0,
        "sigma_L": 0.010,
        "time_opt": 20,
        "time_penalty": 0.16,
        "bonus": 0.0,
    },
    "B": {
        "base": 42.0,
        "amp": 28.0,
        "T_opt": 98.0,
        "L_opt": 0.12,
        "sigma_T": 5.5,
        "sigma_L": 0.010,
        "time_opt": 30,
        "time_penalty": 0.15,
        "bonus": 0.5,
    },
    "C": {
        "base": 46.0,
        "amp": 38.0,
        "T_opt": 125.0,
        "L_opt": 0.08,
        "sigma_T": 4.5,
        "sigma_L": 0.008,
        "time_opt": 40,
        "time_penalty": 0.14,
        "bonus": 2.5,
    },
}


def evaluate_repaired_design(repaired_design):
    p = SOLVENT_PARAMS[repaired_design["solvent"]]
    T = float(repaired_design["temperature"])
    L = float(repaired_design["loading"])
    t = int(repaired_design["time"])
    solvent = repaired_design["solvent"]

    continuous_peak = p["amp"] * np.exp(
        -0.5 * ((T - p["T_opt"]) / p["sigma_T"]) ** 2
        -0.5 * ((L - p["L_opt"]) / p["sigma_L"]) ** 2
    )

    time_term = -p["time_penalty"] * (t - p["time_opt"]) ** 2
    interaction_term = 0.8 * np.sin(T / 9.0) * np.cos(35.0 * L)

    # Deliberately make common repaired-edge regions mediocre.
    edge_penalty = 0.0
    if solvent == "A" and T == 85.0:
        edge_penalty -= 4.0
    if solvent == "B" and (T == 88.0 or T == 108.0 or L == 0.13):
        edge_penalty -= 2.5
    if solvent == "C" and (T == 118.0 or T == 132.0 or L == 0.06 or L == 0.10):
        edge_penalty -= 1.2

    ridge_term = {
        "A": -0.020 * (T - 72.0) * (L - 0.09) * 100.0,
        "B": 0.015 * (T - 96.0) * (L - 0.11) * 100.0,
        "C": 0.025 * (T - 122.0) * (0.09 - L) * 100.0,
    }[solvent]

    score = p["base"] + p["bonus"] + continuous_peak + time_term + interaction_term + ridge_term + edge_penalty
    return float(score)


def objective_mixed_raw(X_raw):
    if X_raw.ndim == 1:
        X_raw = X_raw.unsqueeze(0)

    scores = []
    for x_raw in X_raw:
        decoded = decode_single_design(x_raw)
        repaired = repair_single_design(decoded)
        scores.append(evaluate_repaired_design(repaired))

    return torch.tensor(scores, dtype=torch.double).unsqueeze(-1)


def batch_to_dataframe(X_raw, Y=None):
    if X_raw.ndim == 1:
        X_raw = X_raw.unsqueeze(0)

    rows = []
    Y_vals = None if Y is None else Y.detach().cpu().view(-1).tolist()

    for i, x_raw in enumerate(X_raw):
        x_np = x_raw.detach().cpu().numpy().reshape(-1)
        decoded = decode_single_design(x_raw)
        repaired = repair_single_design(decoded)

        row = {
            "temp_raw": float(x_np[0]),
            "loading_raw": float(x_np[1]),
            "time_raw": float(x_np[2]),
            "solvent_raw": float(x_np[3]),
            "decoded_temperature": decoded["temperature"],
            "decoded_loading": decoded["loading"],
            "decoded_time": decoded["time"],
            "decoded_solvent": decoded["solvent"],
            "repaired_temperature": repaired["temperature"],
            "repaired_loading": repaired["loading"],
            "repaired_time": repaired["time"],
            "repaired_solvent": repaired["solvent"],
            "was_repaired": repaired["was_repaired"],
            "repair_count": repaired["repair_count"],
            "repair_flags": ", ".join(repaired["repair_flags"]) if repaired["repair_flags"] else "None",
        }

        if Y_vals is not None:
            row["score"] = Y_vals[i]

        rows.append(row)

    return pd.DataFrame(rows)

## 5. Generating and inspecting the initial mixed-variable dataset

With the mixed-variable objective now fully defined, we can generate the initial dataset that BO will start from.

As in earlier tutorials, this initial dataset is produced using a **Sobol design** in the latent BO space.
However, because this notebook works with a constrained mixed-variable problem, inspecting the initial dataset is especially important.

That is because the raw Sobol points are sampled in the latent unit box, not directly in the feasible experimental space.

So the initial workflow is:

1. draw Sobol points in the latent space,
2. evaluate them through the full objective pipeline,
3. inspect how they decode and repair,
4. and understand what kind of feasible experimental designs BO is actually starting from.

---

### What the code does

The first line resets the random seed:

```python
torch.manual_seed(seed)
```

This ensures that the initial Sobol design is reproducible.

The next line draws `n_init` Sobol samples in the latent unit box:

```python
train_X_init = draw_sobol_samples(bounds=latent_bounds, n=1, q=n_init).squeeze(0)
```

So `train_X_init` contains the raw latent initial design.

Then:

```python
train_Y_init = objective_mixed_raw(train_X_init)
```

evaluates each latent point through the full pipeline:

> **decode → repair → score**

So `train_Y_init` is not the score of the raw latent points directly.
It is the score of the **repaired feasible catalyst designs** that those latent points correspond to.

The dataframe:

```python
initial_df = batch_to_dataframe(train_X_init, train_Y_init)
```

then makes the initial design interpretable by showing, for each point:

- the raw latent values,
- the decoded experimental values,
- the repaired experimental values,
- the repair flags,
- and the final score.

So this cell gives us both:
- the initial tensors used by BO,
- and a readable experimental summary of what those tensors actually mean.

---

### Why this inspection step matters here

In a standard continuous BO problem, an initial Sobol design is usually already easy to interpret:

- the points lie in the actual search box,
- and no further translation is needed.

Here, that is no longer true.

A latent initial point may decode into an infeasible catalyst condition, which is then repaired into a different feasible one before scoring.

So inspecting the initial dataframe is not just a convenience.
It is part of understanding what BO is really starting from in a constrained mixed-variable setting.

---

### Why the fraction repaired is 1.0

The line

```python
print("Fraction repaired in initial dataset:", float(initial_df["was_repaired"].mean()))
```

computes the fraction of initial Sobol points that required at least one repair.

If the result is:

```python
Fraction repaired in initial dataset: 1.0
```

that means **every single initial latent point** required repair before it became a feasible experimental design.

This may look striking at first, but in the current toy problem it makes sense.

There are two main reasons.

#### 1. The feasible experimental regions are intentionally narrow
The solvent-specific feasible windows are quite restrictive.

For example:

- solvent A only allows a relatively narrow temperature range and only two time levels,
- solvent B only allows a narrower loading interval and only two time levels,
- solvent C occupies a fairly narrow high-temperature region with its own loading and time restrictions.

So the feasible design space is much smaller than the full latent unit box after decoding.

That means a random or quasi-random latent point is quite likely to decode into something that violates at least one of those rules.

#### 2. The repair process includes grid snapping
Even if a decoded point is already roughly compatible with a solvent’s feasible region, it may still be snapped to:

- the nearest allowed time level for that solvent,
- the nearest 5 °C temperature grid,
- or the nearest 0.01 loading grid.

So a point can count as “repaired” even if it was close to feasible already, because the final experimental setting is still adjusted to a more realistic discrete or gridded condition.

So a repair fraction of `1.0` does **not** necessarily mean every point was wildly infeasible.
It means every point underwent at least one adjustment somewhere in the decode–repair pipeline.

---

### Why this is actually useful for the tutorial

A fully repaired initial dataset is useful here because it makes the constrained nature of the problem very visible from the start.

It shows that this is not a problem where BO is exploring a simple free continuous box.
Instead, it is operating in a setting where:

- many latent proposals do not correspond directly to runnable experiments,
- repair is a routine and central part of the modelling process,
- and the feasible experimental space is much smaller and more structured than the latent proposal space.

That is exactly the behaviour this tutorial is trying to illustrate.

So the fact that the initial repair fraction is `1.0` is not a failure of the setup.
It is part of what makes the mixed-variable constrained problem meaningfully different from the earlier BO tutorials.

---

### What the printed outputs tell us

The printed output confirms three useful facts:

- the initial latent design has shape `(n_init, 4)`,
- the score tensor has shape `(n_init, 1)`,
- and the best observed score in the initial dataset is available before BO begins.

So this cell establishes the baseline from which the later BO loops will improve.

---

### Key takeaway

This cell generates the **initial Sobol design in latent space**, evaluates it through the full **decode → repair → score** pipeline, and displays the resulting experimental designs in a readable table.

The fact that the **fraction repaired is 1.0** means that every initial latent point required at least one adjustment before becoming a valid experimental condition.

In this notebook, that makes sense because:

- the solvent-specific feasible regions are deliberately narrow,
- and the repair logic also includes snapping to realistic experimental grids.

So the initial dataset already shows, very clearly, that BO is operating over a constrained mixed-variable experimental design problem rather than a simple continuous box.

In [ ]:
torch.manual_seed(seed)

train_X_init = draw_sobol_samples(bounds=latent_bounds, n=1, q=n_init).squeeze(0)
train_Y_init = objective_mixed_raw(train_X_init)

print("Initial latent design shape:", tuple(train_X_init.shape))
print("Initial score shape:", tuple(train_Y_init.shape))
print("Initial best observed score:", float(torch.max(train_Y_init)))

initial_df = batch_to_dataframe(train_X_init, train_Y_init)
display(initial_df)

print("Fraction repaired in initial dataset:", float(initial_df["was_repaired"].mean()))

## 6. Defining a repair-aware mixed-variable BO loop

This cell defines the first full Bayesian Optimisation loop used in the notebook: a **repair-aware** version of sequential BO.

At a high level, this loop is still very similar to the standard BO loops used earlier in the series.

It still:

1. starts from an initial dataset,
2. fits a Gaussian Process surrogate,
3. constructs an acquisition function,
4. optimises that acquisition function to obtain a candidate,
5. evaluates the objective,
6. updates the dataset,
7. and repeats.

So the core BO mechanics have **not** changed.

What changes here is **how a proposed candidate is treated before it is accepted into the optimisation trajectory**.

That is exactly what makes this loop “aware.”

---

### Why a repair-aware loop is introduced first

In this notebook, BO operates in a **latent continuous space**, but the actual experiments live in the **decoded and repaired experimental space**.

That means two different raw latent proposals can still collapse onto the **same repaired feasible design** after:

> **decode → repair**

This is the key issue.

If we want to take the constrained mixed-variable structure seriously, it is natural to introduce the repair-aware loop first, because it makes the modelling objective explicit from the beginning:

> **the meaningful notion of novelty is not only a new latent point, but a new repaired feasible experiment.**

So this loop is presented first because it is the most problem-aware version of BO for the design space defined in this notebook.

---

### What makes this loop “aware”

The main idea is simple.

After `optimize_acqf(...)` returns a raw latent candidate, the loop immediately converts it into the repaired experimental design that would actually be run:

```python
decoded = decode_single_design(cand.squeeze(0))
repaired = repair_single_design(decoded)
key = repaired_design_key(repaired)
```

So instead of treating the raw latent candidate as the thing that matters, the loop asks whether the **repaired feasible design** has already been seen.

Then it checks:

```python
if key not in seen_keys:
```

A candidate is accepted only if it corresponds to a **new repaired feasible design**.

If it would repair to something already seen, it is rejected and the loop tries again.

That is the central change.

---

### `build_seen_repaired_keys(X_raw)`

This helper function takes an existing latent dataset and converts every point into a repaired-design key.

So for each raw point in the current dataset, it performs:

1. decode,
2. repair,
3. convert to a compact repaired-design key,

and stores that key in a Python set.

This is useful because, at any point in the BO loop, we want to know which repaired feasible designs have already been visited.

So this helper builds the initial `seen_keys` set from the starting dataset.

---

### Why this matters in a mixed-variable constrained problem

In a standard continuous BO problem, two different proposed points are usually treated as meaningfully different experiments.

Here, that is no longer guaranteed.

Because of:

- discrete time snapping,
- categorical solvent decoding,
- solvent-specific feasibility repair,
- and grid snapping of repaired continuous variables,

many different latent points can collapse onto the **same repaired catalyst setting**.

So the repair-aware loop is designed to prevent BO from wasting iterations on raw latent proposals that would ultimately correspond to experiments already explored in repaired form.

---

### Duplicate repaired-design rejections

The variable

```python
duplicate_rejections
```

counts how many times, within the current BO step, the optimiser proposed candidates that repaired to designs already seen before.

This is a very useful diagnostic.

It tells us how often the latent BO proposal mechanism would have led to redundant experiments if this repair-awareness check had not been present.

So this variable helps quantify the many-to-one nature of the latent-to-repaired mapping.

Later in the notebook, this becomes one of the clearest ways to see how the constrained mixed-variable structure changes BO behaviour.

---

### Why multiple attempts are allowed

The loop does not stop after one rejected candidate.

Instead, it allows up to:

- `max_attempts` acquisition-based attempts
- and, if needed, `max_fallback_attempts` Sobol fallback attempts

This is necessary because once duplicate repaired designs are disallowed, the first acquisition maximiser may often be unusable.

So the loop needs a way to keep searching until it finds a candidate whose repaired design is genuinely new.

This is a practical implementation of repair-aware selection rather than a mathematically exact constrained acquisition method.

That is fine for the purposes of this tutorial.

---

### Why a fallback Sobol proposal is included

If repeated acquisition-optimised attempts still keep collapsing onto already-seen repaired designs, the code falls back to:

```python
draw_sobol_samples(...)
```

This is simply a robust backup mechanism.

It ensures that the loop can still explore the latent space and continue even if the acquisition optimiser keeps suggesting repaired duplicates.

So the fallback is not the main logic, but it helps make the repair-aware loop practical and stable.

---

### What stays the same

It is important to note what **has not** changed.

The repair-aware loop still uses:

- the same initial dataset,
- the same GP surrogate,
- the same `LogExpectedImprovement` acquisition function,
- the same latent search space,
- and the same repaired objective function.

So this is **not** a completely different BO algorithm.

It is still standard sequential BO in latent space.

What changes is simply that the loop is now aware of whether a newly proposed latent point would correspond to a repaired feasible design that has already been explored.

That is why this function should be understood as a **minimal but meaningful modification** to standard BO, rather than a brand-new framework.

---

### What gets stored in history

The history stores:

- the current dataset,
- the candidate in raw, decoded, and repaired form,
- the current best point and best score,
- GP hyperparameters,
- and the acquisition value.

In addition, the repair-aware loop also stores:

- `attempts`
- `duplicate_rejections`

These are new and important, because they let us later analyse:

- how often repair-aware BO had to reject repeated repaired designs,
- and how much extra search effort was needed to find genuinely new repaired experiments.

So the history here is specifically designed to support the later comparison against the simpler baseline loop introduced afterward.

---

### Updating `seen_keys`

After a new candidate is finally accepted and evaluated, its repaired-design key is added to:

```python
seen_keys
```

This ensures that future iterations recognise it as already visited.

So the repair-aware memory grows over time as BO explores more of the feasible repaired design space.

That is the mechanism that makes the loop progressively more informed about what it has already effectively tested.

---

### Why this loop is important

This function captures the main methodological idea of the notebook.

It answers the question:

> **What changes if BO is made aware of the repaired experimental designs it has already visited, rather than only the raw latent points it has proposed?**

That is exactly the right question for a mixed-variable constrained BO setting.

By introducing this loop first, the notebook presents the most design-aware BO workflow before later introducing the simpler naïve baseline for comparison.

---

### Key takeaway

This function defines a **repair-aware sequential BO loop**.

It is structurally the same as the earlier BO loops, but with one important addition:

> a proposed latent candidate is accepted only if its **repaired experimental design** has not already been seen before.

So the BO loop is no longer only aware of raw latent coordinates.
It is aware of the **feasible repaired designs** that those coordinates correspond to.

That is the central change that makes this version “aware BO.”This cell defines the second BO loop used in the notebook: a **repair-aware** version of sequential BO.

At a high level, this loop is still very similar to the standard BO loops used earlier in the series.

It still:

1. starts from an initial dataset,
2. fits a Gaussian Process surrogate,
3. constructs an acquisition function,
4. optimises that acquisition function to obtain a candidate,
5. evaluates the objective,
6. updates the dataset,
7. and repeats.

So the core BO mechanics have **not** changed.

What changes here is **how a proposed candidate is treated before it is accepted into the optimisation trajectory**.

That is exactly what makes this loop “aware.”

---

### Why a repair-aware loop is needed

In this notebook, BO operates in a **latent continuous space**, but the actual experiments live in the **decoded and repaired experimental space**.

That means two different raw latent proposals can still collapse onto the **same repaired feasible design** after:

> **decode → repair**

This is the key issue.

A standard latent-space BO loop would treat those two raw points as different proposals, even if they would ultimately correspond to the same actual experiment.

That can waste optimisation steps.

So the purpose of the repair-aware loop is to make BO aware of the fact that:

> **different latent points may represent the same repaired experimental condition**

and to avoid repeatedly accepting such duplicates.

---

### What has changed relative to the naïve BO loop

Structurally, this function is almost the same as the naïve BO loop defined earlier.

It still:

- clones the initial dataset,
- fits a `SingleTaskGP`,
- uses `LogExpectedImprovement`,
- proposes one candidate at a time,
- and stores a rich history.

So the main change is **not** in the GP model, **not** in the acquisition function, and **not** in the BO loop skeleton.

The main change is that this loop now keeps track of the **set of repaired designs that have already been seen**, and uses that information when deciding whether to accept a candidate.

That is the crucial new idea.

---

### `build_seen_repaired_keys(X_raw)`

This helper function takes an existing latent dataset and converts every point into a repaired-design key.

So for each raw point in the current dataset, it performs:

1. decode,
2. repair,
3. convert to a compact repaired-design key,

and stores that key in a Python set.

This is useful because, at any point in the BO loop, we want to know which repaired feasible designs have already been visited.

So this helper builds the initial `seen_keys` set from the starting dataset.

---

### The main idea of “awareness”

Inside the loop, the candidate proposal process changes in the following way.

After `optimize_acqf(...)` returns a raw latent candidate, the loop now immediately does:

```python
decoded = decode_single_design(cand.squeeze(0))
repaired = repair_single_design(decoded)
key = repaired_design_key(repaired)
```

So instead of treating the raw candidate as the thing that matters, the loop converts it into the **repaired experimental design** that would actually be run.

Then it checks:

```python
if key not in seen_keys:
```

This is the crucial change.

A candidate is accepted only if it corresponds to a **new repaired feasible design**.

If it would repair to something already seen, it is rejected and the loop tries again.

That is what makes the BO loop repair-aware.

---

### Why this is different from naïve BO

The naïve loop would accept a candidate as long as it is a new raw latent point proposed by the acquisition optimiser.

So the naïve logic is effectively:

> “this raw point is new, so accept it.”

The repair-aware logic is different:

> “this repaired experimental design is new, so accept it.”

That is a much more meaningful notion of novelty in a constrained mixed-variable optimisation problem.

Because the experimental design is what matters physically, not the latent coordinate by itself.

---

### Duplicate repaired-design rejections

The variable

```python
duplicate_rejections
```

counts how many times, within the current BO step, the optimiser proposed candidates that repaired to designs already seen before.

This is a very useful diagnostic.

It tells us how often the latent BO proposal mechanism would have led to redundant experiments if this repair-awareness check had not been present.

So this variable helps quantify the many-to-one nature of the latent-to-repaired mapping.

Later in the notebook, this becomes one of the clearest ways to see why the repair-aware loop is different from the naïve one.

---

### Why multiple attempts are allowed

The loop does not stop after one rejected candidate.

Instead, it allows up to:

- `max_attempts` acquisition-based attempts
- and, if needed, `max_fallback_attempts` Sobol fallback attempts

This is necessary because once duplicate repaired designs are disallowed, the first acquisition maximiser may often be unusable.

So the loop needs a way to keep searching until it finds a candidate whose repaired design is genuinely new.

This is a practical implementation of repair-aware selection rather than a mathematically exact constrained acquisition method.

That is fine for the purposes of this tutorial.

---

### Why a fallback Sobol proposal is included

If repeated acquisition-optimised attempts still keep collapsing onto already-seen repaired designs, the code falls back to:

```python
draw_sobol_samples(...)
```

This is simply a robust backup mechanism.

It ensures that the loop can still explore the latent space and continue even if the acquisition optimiser keeps suggesting repaired duplicates.

So the fallback is not the main logic, but it helps make the repair-aware loop practical and stable.

---

### What stays the same

It is important to note what **has not** changed.

The repair-aware loop still uses:

- the same initial dataset,
- the same GP surrogate,
- the same `LogExpectedImprovement` acquisition function,
- the same latent search space,
- and the same repaired objective function.

So this is **not** a completely different BO algorithm.

It is still standard sequential BO in latent space.

What changes is simply that the loop is now aware of whether a newly proposed latent point would correspond to a repaired feasible design that has already been explored.

That is why this function should be understood as a **minimal but meaningful modification** to standard BO, rather than a brand-new framework.

---

### What gets stored in history

As with the naïve loop, the history stores:

- the current dataset,
- the candidate in raw, decoded, and repaired form,
- the current best point and best score,
- GP hyperparameters,
- and the acquisition value.

In addition, the repair-aware loop also stores:

- `attempts`
- `duplicate_rejections`

These are new and important, because they let us later analyse:

- how often repair-aware BO had to reject repeated repaired designs,
- and how much extra search effort was needed to find genuinely new repaired experiments.

So the history here is specifically designed to support the later comparison against the naïve loop.

---

### Updating `seen_keys`

After a new candidate is finally accepted and evaluated, its repaired-design key is added to:

```python
seen_keys
```

This ensures that future iterations recognise it as already visited.

So the repair-aware memory grows over time as BO explores more of the feasible repaired design space.

That is the mechanism that makes the loop progressively more informed about what it has already effectively tested.

---

### Why this loop is important

This function is the key methodological contribution of the notebook.

It answers the question:

> **What changes if BO is made aware of the repaired experimental designs it has already visited, rather than only the raw latent points it has proposed?**

That is exactly the right question for a mixed-variable constrained BO setting.

Without this loop, the notebook would only show standard latent BO on a repaired objective.
With this loop, it can show why explicitly reasoning in terms of **repaired feasible designs** can change optimisation behaviour in a meaningful way.

---

### Key takeaway

This function defines a **repair-aware sequential BO loop**.

It is structurally the same as the earlier BO loops, but with one important addition:

> a proposed latent candidate is accepted only if its **repaired experimental design** has not already been seen before.

So the BO loop is no longer only aware of raw latent coordinates.
It is aware of the **feasible repaired designs** that those coordinates correspond to.

That is the central change that makes this version “aware BO.”

In [ ]:
def build_seen_repaired_keys(X_raw):
    seen = set()
    for x_raw in X_raw:
        decoded = decode_single_design(x_raw)
        repaired = repair_single_design(decoded)
        seen.add(repaired_design_key(repaired))
    return seen

def run_repair_aware_mixed_bo_loop(
    train_X_init,
    train_Y_init,
    objective_fn,
    bounds,
    n_steps=20,
    num_restarts=20,
    raw_samples=256,
    max_attempts=40,
    max_fallback_attempts=200,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()
    seen_keys = build_seen_repaired_keys(train_X)
    history = []

    for step in range(n_steps + 1):
        gp = SingleTaskGP(
            train_X=train_X,
            train_Y=train_Y,
            input_transform=Normalize(d=train_X.shape[-1]),
            outcome_transform=Standardize(m=1),
        )
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        gp.eval()

        best_f = float(torch.max(train_Y))
        acq = LogExpectedImprovement(model=gp, best_f=best_f)

        accepted = False
        attempts = 0
        duplicate_rejections = 0
        candidate = None
        candidate_decoded = None
        candidate_repaired = None
        acq_value_out = None

        for _ in range(max_attempts):
            attempts += 1
            cand, acq_value = optimize_acqf(
                acq_function=acq,
                bounds=bounds,
                q=1,
                num_restarts=num_restarts,
                raw_samples=raw_samples,
            )

            decoded = decode_single_design(cand.squeeze(0))
            repaired = repair_single_design(decoded)
            key = repaired_design_key(repaired)

            if key not in seen_keys:
                candidate = cand
                candidate_decoded = decoded
                candidate_repaired = repaired
                acq_value_out = float(acq_value)
                accepted = True
                break
            else:
                duplicate_rejections += 1

        if not accepted:
            for _ in range(max_fallback_attempts):
                attempts += 1
                cand = draw_sobol_samples(bounds=bounds, n=1, q=1).squeeze(0)
                decoded = decode_single_design(cand.squeeze(0))
                repaired = repair_single_design(decoded)
                key = repaired_design_key(repaired)

                if key not in seen_keys:
                    candidate = cand
                    candidate_decoded = decoded
                    candidate_repaired = repaired
                    acq_value_out = np.nan
                    accepted = True
                    break
                else:
                    duplicate_rejections += 1

        if not accepted:
            raise RuntimeError("Constraint-aware loop failed to find a new repaired design.")

        best_idx = torch.argmax(train_Y)

        history.append({
            "step": step,
            "train_X": train_X.clone(),
            "train_Y": train_Y.clone(),
            "candidate_raw": candidate.clone(),
            "candidate_decoded": dict(candidate_decoded),
            "candidate_repaired": dict(candidate_repaired),
            "candidate_was_repaired": candidate_repaired["was_repaired"],
            "candidate_acq_value": acq_value_out,
            "attempts": attempts,
            "duplicate_rejections": duplicate_rejections,
            "best_x": train_X[best_idx].clone(),
            "best_y": train_Y[best_idx].clone(),
            "lengthscale": gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy().copy(),
            "noise": gp.likelihood.noise.detach().cpu().item(),
        })

        if step < n_steps:
            y_new = objective_fn(candidate)
            train_X = torch.cat([train_X, candidate], dim=0)
            train_Y = torch.cat([train_Y, y_new], dim=0)
            seen_keys.add(repaired_design_key(candidate_repaired))

    return history

## 7. Running the repair-aware mixed-variable BO loop

Now that the repair-aware BO loop has been defined, we can run it on the initial latent dataset and inspect the resulting optimisation trajectory.

This cell is the first full optimisation run in the notebook.

So at this point, the modelling pipeline is fully active:

> **latent BO proposal → decode → repair → evaluate**

and the loop is using repaired-design awareness to avoid repeatedly accepting candidates that would collapse onto already-seen feasible experiments.

---

### What the code does

The cell begins by resetting the random seed:

```python
torch.manual_seed(seed)
```

This helps keep the run reproducible.

The repair-aware BO loop is then executed through:

```python
history_aware = run_repair_aware_mixed_bo_loop(...)
```

using:

- the initial latent dataset `train_X_init`,
- the corresponding initial scores `train_Y_init`,
- the repaired mixed-variable objective `objective_mixed_raw`,
- the latent bounds,
- and the chosen optimisation settings such as `n_steps`, `num_restarts`, and `raw_samples`.

So this line runs the full repair-aware BO optimisation from start to finish.

---

### What the printed outputs mean

The output lines provide a few basic checks:

#### `Number of stored BO states`
This should be `n_steps + 1`, because the history includes:

- the initial state before any new BO update,
- plus one stored state for each BO iteration.

#### `Final number of observations`
This shows how many total latent observations are in the dataset at the end of the run.

Since the loop starts from `n_init` points and then adds one new point per BO step, this should equal:

$begin:math:display$
n\_\{\\text\{init\}\} \+ n\_\{\\text\{steps\}\}
$end:math:display$

#### `Final best observed score`
This is the best repaired catalyst score found by the repair-aware BO loop by the end of the run.

That is one of the main summary quantities of interest.

---

### Why we decode and repair the final best point again

The code then takes:

```python
aware_best_raw = history_aware[-1]["best_x"]
```

which is the raw latent point corresponding to the best score seen so far.

But since a raw latent point is not directly interpretable as an experiment, the notebook decodes and repairs it again:

```python
aware_best_decoded = decode_single_design(aware_best_raw)
aware_best_repaired = repair_single_design(aware_best_decoded)
```

This is important because the real outcome we care about is not the latent point itself, but the **repaired feasible catalyst design** that it corresponds to.

So this final print statement gives the reader the best design in a user-facing, experimentally meaningful form.

---

### Why `repair_count` can be 5

One detail that may stand out is that the best repaired design can show:

```python
"repair_count": 5
```

This simply means that **five separate repair actions were applied** when converting the decoded best latent point into its final feasible catalyst design.

That does **not** mean the design was repaired five times in a repeated sense.
It means that, within one pass through `repair_single_design(...)`, five distinct repair rules were triggered.

For example, in the printed repaired design here, the repair flags are:

- `high_temp_requires_C`
- `C_temp_cap`
- `C_time_window`
- `temp_grid_snap`
- `loading_grid_snap`

So the total repair count is 5 because all five of those adjustments occurred.

---

### Interpreting those five repairs

These five repairs arise from different parts of the repair pipeline:

#### 1. `high_temp_requires_C`
The decoded temperature was above the threshold where only solvent C is allowed, so the solvent had to be changed to C.

#### 2. `C_temp_cap`
After the solvent became C, the decoded temperature was still above solvent C’s allowed window, so it had to be clipped down to the C-compatible maximum.

#### 3. `C_time_window`
The decoded time was not in solvent C’s allowed set `{40, 50}`, so it was snapped to the nearest valid C time level.

#### 4. `temp_grid_snap`
After constraint repair, the temperature was then snapped to the nearest 5 °C grid point to reflect realistic experimental control.

#### 5. `loading_grid_snap`
Similarly, the loading was snapped to the nearest 0.01 grid point.

So the repair count is simply the number of distinct corrections needed to transform the decoded latent proposal into a valid, gridded, experimentally meaningful design.

---

### Why this is not a problem

A repair count of 5 is not an error or a sign that something went wrong.

In fact, it is quite consistent with the design of this notebook.

The whole point of the toy problem is that:

- the latent BO space is broad and continuous,
- the experimental design space is narrower and structured,
- and many raw latent proposals require substantial repair before becoming runnable experiments.

So a relatively large repair count here simply reflects the fact that the best latent point lies in a region that only becomes valid after several solvent-specific and grid-based adjustments.

That is exactly the kind of behaviour the notebook is intended to illustrate.

---

### What this tells us about the problem structure

This result reinforces an important lesson of the notebook:

> the raw latent BO candidate and the actual repaired experimental design are not the same object.

In a mixed-variable constrained setting, a high-quality repaired design can be reached from a latent proposal that initially violates several feasibility rules.

That is why decoding and repair are not cosmetic post-processing steps.
They are central to how the optimisation problem is actually defined.

---

### Key takeaway

This cell runs the **repair-aware mixed-variable BO loop** and summarises the best repaired catalyst design found.

The `repair_count` value counts how many distinct repair rules were triggered when converting the decoded best latent point into its final feasible design.

So if the output shows:

```python
"repair_count": 5
```

that means five separate adjustments were applied — for example a solvent switch, a temperature clip, a time correction, and grid snapping steps — before the final experimental design was obtained.

This is a natural consequence of optimising in latent space while evaluating performance in a repaired constrained experimental space.

In [ ]:
torch.manual_seed(seed)

history_aware = run_repair_aware_mixed_bo_loop(
    train_X_init=train_X_init,
    train_Y_init=train_Y_init,
    objective_fn=objective_mixed_raw,
    bounds=latent_bounds,
    n_steps=n_steps,
    num_restarts=20,
    raw_samples=256,
)

print("Number of stored BO states:", len(history_aware))
print("Final number of observations:", history_aware[-1]["train_X"].shape[0])
print("Final best observed score:", float(torch.max(history_aware[-1]["train_Y"])))

aware_best_raw = history_aware[-1]["best_x"]
aware_best_decoded = decode_single_design(aware_best_raw)
aware_best_repaired = repair_single_design(aware_best_decoded)
print("Repair-aware best repaired design:", aware_best_repaired)

## 8. Defining a naïve mixed-variable BO loop

This cell defines the first full Bayesian Optimisation loop used in the notebook.

Structurally, this loop is **the same kind of reusable BO loop we have already defined in previous tutorials**.

In particular, it follows the familiar pattern:

1. start from an initial dataset,
2. fit a Gaussian Process surrogate,
3. build an acquisition function from the current surrogate,
4. optimise that acquisition function to choose the next candidate,
5. evaluate the objective there,
6. update the dataset,
7. and repeat.

So at the BO-mechanics level, there is nothing fundamentally new here.

The main difference is **not the BO loop structure itself**, but the kind of objective it is being applied to.

Earlier tutorials used:

- low-dimensional continuous objectives,
- higher-dimensional continuous objectives,
- or standard batch BO settings.

Here, the loop is being applied to a **mixed-variable constrained objective** in which each raw latent proposal is later:

> **decoded → repaired → evaluated**

So this function should be understood as:

> **the standard sequential BO loop, now applied to a latent-space representation of a mixed-variable constrained problem**

---

### Why this loop is called “naïve”

The key reason this loop is called **naïve** is that it does **not** explicitly reason about whether different raw latent proposals may collapse to the same repaired feasible design.

That is an important point.

This loop does store:

- the decoded candidate,
- the repaired candidate,
- and whether that candidate was repaired,

but it does **not** use that information to decide whether a candidate should be accepted.

So if BO proposes two different raw latent points that both repair to the same feasible catalyst condition, this loop will still treat them as separate iterations in the optimisation history.

That is why it is “naïve” relative to the more repair-aware loop introduced later.

---

### What the code does

The function begins by cloning the initial dataset:

```python
train_X = train_X_init.clone()
train_Y = train_Y_init.clone()
```

This is standard good practice and ensures that the original initial design is not modified outside the function.

A history list is then created to store the full BO trajectory over time.

---

### The BO loop itself

The main loop runs over:

```python
for step in range(n_steps + 1):
```

As in previous tutorials, this means the history will include:

- the initial state,
- plus one stored BO state for each optimisation step,
- with the final stored state containing a newly proposed candidate that has not yet been appended if `step == n_steps`

So this history structure is consistent with the earlier notebooks.

---

### Fitting the GP surrogate

At each iteration, the loop fits a standard `SingleTaskGP` model:

```python
gp = SingleTaskGP(
    train_X=train_X,
    train_Y=train_Y,
    input_transform=Normalize(d=train_X.shape[-1]),
    outcome_transform=Standardize(m=1),
)
```

This is the same modelling pattern used in previous tutorials:

- inputs are normalised,
- outputs are standardised,
- and the GP is refit at every BO step.

So from the surrogate-modelling point of view, this loop is entirely familiar.

---

### Building and optimising the acquisition function

The loop then defines:

```python
best_f = float(torch.max(train_Y))
acq = LogExpectedImprovement(model=gp, best_f=best_f)
```

and uses `optimize_acqf(...)` to choose the next candidate.

Again, this is the same standard sequential BO pattern used before:

- compute the current best observed value,
- define a one-point acquisition function,
- and optimise it over the latent search box.

So the acquisition logic itself is unchanged.

---

### What is new here

What is new is what happens **after** the candidate is chosen.

The raw candidate is interpreted through:

```python
candidate_decoded = decode_single_design(candidate.squeeze(0))
candidate_repaired = repair_single_design(candidate_decoded)
```

This is important because, in this notebook, the BO candidate does not directly correspond to a valid experiment.
It must first be:

- decoded into user-facing variables,
- then repaired into a feasible catalyst setting.

So this loop stores both the decoded and repaired forms of the candidate, even though it does not yet use them for decision-making.

That is a useful diagnostic step, because it lets us later inspect what kinds of experiments the naïve BO loop was effectively proposing.

---

### What gets stored in `history`

At each step, the loop stores:

- the current step index,
- the current dataset,
- the raw candidate,
- the decoded candidate,
- the repaired candidate,
- whether repair occurred,
- the acquisition value,
- the current best point and best score,
- the learned GP lengthscales,
- and the learned GP noise.

So the history is intentionally rich.

This is helpful because later sections of the notebook will use it to compare the naïve loop against the repair-aware loop in terms of:

- optimisation progress,
- repaired-design behaviour,
- and the best final catalyst setting found.

---

### Updating the dataset

If the current iteration is not the final stored state, the loop evaluates the candidate through:

```python
y_new = objective_fn(candidate)
```

This is important.

The objective function here is the mixed-variable latent-space wrapper defined earlier, which means:

- the raw latent candidate is decoded,
- repaired,
- and then scored.

So even though this is called the **naïve** loop, it is still being evaluated on the **same repaired experimental objective** as the later repair-aware loop.

That is why the difference between the two methods is not in how scores are computed, but in how candidate proposals are treated when they lead to repeated repaired designs.

After evaluation, the new point is appended to the dataset in the usual BO way.

---

### Why this loop is useful

This loop serves as the baseline for the rest of the notebook.

It answers the question:

> **What happens if we run a completely standard sequential BO loop on the latent mixed-variable objective, without explicitly accounting for repeated repaired designs?**

That is an important reference point.

Without this loop, there would be no baseline against which to compare the later repair-aware version.

So this function is not just another BO loop.
It is the baseline standard method against which the constrained-aware improvement will be judged.

---

### Key takeaway

This function defines a **standard sequential BO loop**, structurally the same as the reusable BO loops introduced in earlier tutorials.

The difference is that it is now applied to a **mixed-variable constrained objective** in which every raw latent proposal is later decoded and repaired before evaluation.

It is called **naïve** because it stores the repaired design information for analysis, but does **not** use that information to prevent repeated repaired feasible designs from being proposed again.

In [ ]:
def run_naive_mixed_bo_loop(
    train_X_init,
    train_Y_init,
    objective_fn,
    bounds,
    n_steps=20,
    num_restarts=20,
    raw_samples=256,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()
    history = []

    for step in range(n_steps + 1):
        gp = SingleTaskGP(
            train_X=train_X,
            train_Y=train_Y,
            input_transform=Normalize(d=train_X.shape[-1]),
            outcome_transform=Standardize(m=1),
        )
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        gp.eval()

        best_f = float(torch.max(train_Y))
        acq = LogExpectedImprovement(model=gp, best_f=best_f)

        candidate, acq_value = optimize_acqf(
            acq_function=acq,
            bounds=bounds,
            q=1,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )

        candidate_decoded = decode_single_design(candidate.squeeze(0))
        candidate_repaired = repair_single_design(candidate_decoded)

        best_idx = torch.argmax(train_Y)

        history.append({
            "step": step,
            "train_X": train_X.clone(),
            "train_Y": train_Y.clone(),
            "candidate_raw": candidate.clone(),
            "candidate_decoded": dict(candidate_decoded),
            "candidate_repaired": dict(candidate_repaired),
            "candidate_was_repaired": candidate_repaired["was_repaired"],
            "candidate_acq_value": float(acq_value),
            "best_x": train_X[best_idx].clone(),
            "best_y": train_Y[best_idx].clone(),
            "lengthscale": gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy().copy(),
            "noise": gp.likelihood.noise.detach().cpu().item(),
        })

        if step < n_steps:
            y_new = objective_fn(candidate)
            train_X = torch.cat([train_X, candidate], dim=0)
            train_Y = torch.cat([train_Y, y_new], dim=0)

    return history

## 9. Running the naïve mixed-variable BO loop

With the repair-aware BO loop already defined and run, we now execute the **naïve latent-space BO loop** on exactly the same initial dataset and objective.

This gives us the baseline comparison that the rest of the notebook needs.

The point of this cell is not to introduce a new objective or a new model.
Instead, it is to show what happens if we use an otherwise standard sequential BO loop **without explicitly preventing repeated repaired designs**.

So this cell lets us compare:

- a BO loop that is aware of repaired-design duplication,
- versus a BO loop that simply accepts the next raw latent candidate returned by acquisition optimisation.

That is the central methodological comparison of the notebook.

---

### What the code does

The line

```python
history_naive = run_naive_mixed_bo_loop(...)
```

runs the full naïve BO loop from the same starting dataset used earlier.

So this run uses:

- the same initial latent design `train_X_init`,
- the same initial scores `train_Y_init`,
- the same repaired mixed-variable objective `objective_mixed_raw`,
- the same latent bounds,
- and the same optimisation settings such as `n_steps`, `num_restarts`, and `raw_samples`.

This is important because it makes the comparison fair.

The difference between the two methods is therefore **not** due to different starting points or different BO budgets.
It is due only to how they treat candidate proposals once the latent point has been decoded and repaired.

---

### What the printed outputs mean

As with the repair-aware run, the printed outputs give a few useful checks.

#### `Number of stored BO states`
This should again be `n_steps + 1`, since the history stores:

- the initial state,
- plus one stored state for each BO iteration.

#### `Final number of observations`
This should equal:

$$
n_{\text{init}} + n_{\text{steps}}
$$

because one new candidate is appended at each BO step.

#### `Final best observed score`
This is the best repaired catalyst score found by the naïve BO run by the end of the optimisation.

That is the most important summary quantity for later comparison against the repair-aware loop.

---

### Why we decode and repair the final best point again

The history stores the best latent point as:

```python
naive_best_raw = history_naive[-1]["best_x"]
```

But just as in the repair-aware run, this raw latent point is not yet directly interpretable as an experimental catalyst design.

So the notebook again applies:

```python
naive_best_decoded = decode_single_design(naive_best_raw)
naive_best_repaired = repair_single_design(naive_best_decoded)
```

This gives the final best design in user-facing experimental terms.

That is important because the scientifically meaningful result is the repaired feasible catalyst setting, not the latent coordinates themselves.

So this final print statement gives the best actual design found by the naïve BO method.

---

### Why the naïve best repaired design can still look highly repaired

One interesting feature of the printed output is that the best naïve design may still show a relatively large repair count.

For example, if the output includes something like:

```python
'repair_count': 4
```

that means the decoded best latent point required four distinct repair actions before becoming the final feasible catalyst design.

This is completely consistent with the notebook setup.

The naïve BO loop proposes points in latent space, not directly in the feasible experimental space.
So even a high-scoring final design may be reached through a latent proposal that:

- decodes to an invalid time,
- lies outside a solvent’s temperature window,
- or requires grid snapping before it becomes experimentally meaningful.

So the fact that the best naïve design can still be heavily repaired is not a bug.
It is part of the whole modelling structure.

---

### Why the naïve and repair-aware runs can end with the same best repaired design

It is also possible — and in this notebook it may indeed happen — that the naïve and repair-aware loops end up with the **same final best repaired design** and the **same final best score**.

That is not worrying.

Both methods are still being evaluated on the same repaired experimental objective.
So if one repaired catalyst design is genuinely best, both loops can eventually find it.

The important difference is not necessarily the identity of the final optimum.

The more important difference is often:

- how quickly the method reaches that strong repaired region,
- how many redundant repaired designs are revisited on the way,
- and how efficiently the BO budget is used.

So if the naïve loop eventually reaches the same repaired design but much later than the repair-aware loop, that is already a meaningful and realistic result.

---

### What this cell contributes to the tutorial

This cell completes the two-method setup needed for the rest of the notebook.

After this point, we now have:

- `history_aware`
- and `history_naive`

which can be compared directly in terms of:

- best observed score over BO iterations,
- the number of unique repaired designs explored,
- duplicate repaired-design rejections,
- and the final best catalyst design found by each method.

So this cell is the point where the comparison becomes possible.

---

### Key takeaway

This cell runs the **naïve mixed-variable BO loop** on the same repaired objective and from the same initial dataset as the repair-aware loop.

The naïve loop is still evaluated through the same **decode → repair → score** pipeline, so its final best design is also interpreted in repaired experimental terms.

What distinguishes it is that it does **not** prevent the BO trajectory from revisiting repaired designs that have already effectively been explored.

That makes this run the natural baseline against which the repair-aware method can now be compared.

In [ ]:
history_naive = run_naive_mixed_bo_loop(
    train_X_init=train_X_init,
    train_Y_init=train_Y_init,
    objective_fn=objective_mixed_raw,
    bounds=latent_bounds,
    n_steps=n_steps,
    num_restarts=20,
    raw_samples=256,
)

print("Number of stored BO states:", len(history_naive))
print("Final number of observations:", history_naive[-1]["train_X"].shape[0])
print("Final best observed score:", float(torch.max(history_naive[-1]["train_Y"])))

naive_best_raw = history_naive[-1]["best_x"]
naive_best_decoded = decode_single_design(naive_best_raw)
naive_best_repaired = repair_single_design(naive_best_decoded)
print("Naive best repaired design:", naive_best_repaired)

## 10. Comparing best observed score over BO iterations

We now compare the optimisation trajectories of the two BO strategies by plotting the **best observed score so far** against BO iteration.

This is one of the most direct ways to compare their practical behaviour.

For each stored BO state, the code extracts:

- the best score reached by the **naïve latent BO** loop,
- and the best score reached by the **repair-aware BO** loop,

and then plots those two running-best curves on the same axes.

---

### What the code is doing

The two lists

```python
best_naive = [float(torch.max(h["train_Y"])) for h in history_naive]
best_aware = [float(torch.max(h["train_Y"])) for h in history_aware]
```

collect the best observed score up to each BO iteration.

So each point on the curve answers the question:

> **By this BO iteration, what is the best repaired catalyst design found so far?**

The rest of the cell simply plots those two trajectories for visual comparison.

---

### How to read the figure

Because this is a **running-best** curve:

- an upward jump means BO has found a better repaired design,
- a plateau means recent iterations did not improve the current best score.

So the figure is not showing the score of the most recent candidate.
It is showing the best score accumulated so far.

That makes it a very natural summary of BO progress.

---

### Why this comparison matters

Both methods are optimising the **same repaired objective**.
So in principle, both may eventually reach the same final best repaired design.

But the important question here is not only:

> **Do they end at the same place?**

It is also:

> **How quickly do they get there?**

That is exactly what this plot reveals.

---

### Why repair-aware BO can accelerate the BO iterations

This is the key point of the figure.

The repair-aware BO loop can accelerate progress over BO iterations because it avoids accepting raw latent candidates that would collapse onto repaired feasible designs that have already been seen.

In other words, it tries to make each BO step correspond to a **new repaired experiment**, rather than wasting iterations on latent-space novelty that does not translate into real experimental novelty.

So when the design space is strongly many-to-one under:

> **decode → repair**

the repair-aware loop can use the BO budget more effectively.

That means it may:

- reach good repaired regions earlier,
- escape mediocre repaired basins faster,
- and improve the running-best curve in fewer BO iterations.

This is exactly the kind of acceleration that this notebook is designed to illustrate.

---

### Interpreting the typical behaviour in this figure

In the current result, the repair-aware BO curve improves earlier and reaches the high-scoring repaired region noticeably sooner than the naïve latent BO curve.

That is the central observation.

The naïve BO loop spends more iterations trapped on a lower plateau before eventually catching up.
By contrast, the repair-aware loop reaches the strong repaired design much earlier.

So even if both methods eventually obtain the same final best score, the repair-aware method is still clearly more efficient **in BO iterations**.

That matters because, in realistic optimisation settings, each BO iteration may correspond to:

- another experimental round,
- another synthesis cycle,
- another batch of measurements,
- or another costly decision step.

So faster improvement in terms of BO iteration is already a meaningful practical advantage.

---

### Why the two methods can still end at the same final score

It is worth stating this explicitly.

The fact that the two methods may end at the same final score is **not** a contradiction.

Both loops are still evaluating the same repaired feasible objective.
So if one repaired catalyst design is genuinely best, both loops can eventually find it.

The difference is that the repair-aware loop may get there with:

- fewer wasted BO steps,
- less repeated exploration of equivalent repaired designs,
- and faster convergence in terms of optimisation rounds.

So the practical benefit here is not necessarily a different final optimum, but a better path to that optimum.

---

### What this figure teaches us

This figure makes one of the main lessons of the notebook very concrete:

> **In a mixed-variable constrained BO problem, being aware of repaired-design duplication can accelerate optimisation over BO iterations.**

That is because the real experimental space is the repaired design space, not the raw latent space.

So a loop that reasons in terms of repaired-design novelty can often move through the problem more efficiently than one that only reasons in terms of raw latent proposals.

---

### Key takeaway

This cell compares the running-best score of the naïve and repair-aware BO loops over BO iterations.

The main lesson is that the **repair-aware BO loop can accelerate BO progress** by avoiding repeated repaired feasible designs and therefore using iterations more effectively.

So even when both methods eventually reach the same final best score, the repair-aware version can still be clearly better in terms of **how quickly** that strong repaired design is found.

In [ ]:
best_naive = [float(torch.max(h["train_Y"])) for h in history_naive]
best_aware = [float(torch.max(h["train_Y"])) for h in history_aware]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    range(len(best_naive)),
    best_naive,
    "-o",
    linewidth=2.2,
    label="Naive latent BO",
)
ax.plot(
    range(len(best_aware)),
    best_aware,
    "-o",
    linewidth=2.2,
    label="Repair-aware BO",
)

ax.set_title("Best observed score over BO iterations", fontsize=18, fontweight="bold")
ax.set_xlabel("BO iteration", fontsize=22, fontweight="bold")
ax.set_ylabel("Best observed score", fontsize=22, fontweight="bold")
style_ax(ax)
ax.legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 11. Tracking repaired-design diversity and duplicate repaired-design rejections

The previous best-score comparison showed that the repair-aware BO loop reaches strong repaired designs much earlier than the naïve latent BO loop.

This cell explains *why* that happens by looking more directly at behaviour in the **repaired experimental space**.

It introduces two diagnostics:

- the **cumulative number of unique repaired designs visited**
- the **cumulative number of duplicate repaired-design rejections** in the repair-aware loop

Together, these plots make the main mechanism of the notebook much more explicit.

---

### What the code is doing

The function

```python
def repaired_history_keys(history):
```

extracts the repaired-design key for each accepted candidate in the BO history.

So instead of looking at raw latent points, we now track the sequence of **repaired feasible experiments** that each method effectively explored.

The function

```python
def cumulative_unique_count(keys):
```

then computes how many *distinct* repaired designs have been seen up to each BO iteration.

This is useful because in a constrained mixed-variable problem, different latent points do **not** necessarily correspond to different effective experiments.

So this plot tells us how much genuinely new repaired-design territory each method is covering over time.

For the repair-aware loop, the code also computes:

```python
dup_rejections_aware = np.cumsum([h["duplicate_rejections"] for h in history_aware[:-1]])
```

which tracks how many times the loop had to reject proposed candidates because they would have repaired to a design that had already been seen.

That is the most direct diagnostic of repair-awareness in action.

---

## Left panel: cumulative unique repaired designs visited

This panel compares the cumulative number of *distinct repaired feasible designs* visited by the two BO loops.

### Why the repair-aware curve increases steadily

The repair-aware BO loop is explicitly designed to accept a candidate **only if its repaired design is new**.

That means each accepted BO iteration is, by construction, trying to contribute a genuinely new repaired experiment.

So once the loop accepts a candidate, it almost always pushes the cumulative unique count upward.

That is why the repair-aware curve rises so steadily.

In this notebook, that steady increase means:

> the repair-aware method is continually expanding its coverage of the repaired experimental space, rather than spending many iterations revisiting already-seen repaired settings.

This is exactly the behaviour we want from a method that is aware of repaired-design duplication.

---

### Why the naïve latent BO curve shows several plateaus

The naïve latent BO loop behaves differently.

It accepts whatever raw latent candidate is proposed by acquisition optimisation, even if that raw point decodes and repairs to a design that has effectively already been explored.

So a plateau in the naïve curve means:

> several successive latent BO proposals were different in raw latent space, but they collapsed onto repaired designs that were not genuinely new.

That is a very important point.

The plateau does **not** mean BO stopped proposing new latent points.
It means those new latent points failed to translate into new repaired experiments.

So the plateaus are a direct visual signature of the many-to-one structure of the mapping:

> **latent proposal → decoded design → repaired design**

In other words, the naïve loop keeps moving in latent space, but often fails to make meaningful progress in the repaired experimental space.

That is why its cumulative unique repaired-design count grows more slowly and contains repeated flat regions.

---

### What this tells us

The contrast between the two curves makes the main lesson very clear:

- **repair-aware BO** is exploring the repaired experimental space more efficiently
- **naïve latent BO** is wasting iterations on repaired-design repetition

This explains why the repair-aware method was able to improve the best score more quickly in the earlier plot.

It is not only “searching faster” in some abstract sense.
It is making better use of iterations by turning them into **new repaired experiments** more consistently.

---

## Right panel: cumulative duplicate repaired-design rejections

This panel tracks how many proposed candidates were rejected by the repair-aware loop because they would have repaired to a design already seen before.

### Why the cumulative rejections increase steadily

This curve increases steadily because the repair-aware loop repeatedly encounters latent candidates that are *new as raw points* but *not new as repaired experiments*.

As the BO run proceeds, this becomes increasingly likely.

Why?

Because more and more repaired designs have already been visited, while the feasible repaired design space is relatively structured and limited.

So later in the run, the acquisition optimiser often keeps pointing toward latent regions whose repaired outputs are already known to be attractive.
But since the repair-aware loop forbids revisiting those repaired designs, it has to reject them and keep searching.

That is why the cumulative rejection count keeps rising.

So the steady growth of this curve means:

> the repair-aware loop is repeatedly preventing the optimisation from wasting iterations on repaired duplicates.

This is not a sign that the method is malfunctioning.
It is evidence that the repair-aware mechanism is actively doing useful work.

---

### Why the rejection count becomes large

The large cumulative rejection count reflects the fact that this toy problem has a strongly many-to-one repaired mapping.

Because of:

- solvent switching,
- solvent-specific clipping,
- discrete time snapping,
- and grid snapping of temperature and loading,

many different latent candidates can collapse onto the same repaired design.

So as BO increasingly concentrates on good parts of latent space, it naturally proposes many candidates that repair to already-known high-quality feasible regions.

The repair-aware loop must therefore reject them to force continued exploration of *new* repaired designs.

So a large cumulative rejection count is actually expected in this setting.

It quantifies exactly how strong the repaired-design duplication problem is.

---

## Why these two panels matter together

These two diagnostics complement each other very naturally.

The left panel shows the **effect**:

- repair-aware BO visits more unique repaired designs
- naïve BO often stalls in terms of repaired-design novelty

The right panel shows the **mechanism**:

- repair-aware BO is actively rejecting many candidate proposals that would otherwise have revisited already-seen repaired designs

So together they explain the earlier performance comparison much more clearly than the best-score curve alone.

They show that repair-aware BO is not just “getting lucky.”
It is behaving differently in a principled way because it reasons in terms of repaired experimental novelty.

---

## Key takeaway

This cell makes the central idea of the notebook directly visible in the repaired design space.

The **repair-aware BO** curve rises steadily because that loop is explicitly constructed to accept only candidates whose **repaired designs are new**.

By contrast, the **naïve latent BO** curve shows multiple plateaus because many raw latent candidates collapse onto repaired designs that have already been seen, so latent-space novelty does not always translate into experimental novelty.

The steadily increasing **cumulative duplicate repaired-design rejections** confirm that the repair-aware loop is repeatedly blocking such redundant repaired proposals, which is exactly why it can move through the constrained mixed-variable design space more efficiently.

In [ ]:
def repaired_history_keys(history):
    keys = []
    for h in history[:-1]:
        keys.append(repaired_design_key(h["candidate_repaired"]))
    return keys

def cumulative_unique_count(keys):
    seen = set()
    counts = []
    for key in keys:
        seen.add(key)
        counts.append(len(seen))
    return counts

naive_repaired_keys = repaired_history_keys(history_naive)
aware_repaired_keys = repaired_history_keys(history_aware)

naive_unique_counts = cumulative_unique_count(naive_repaired_keys)
aware_unique_counts = cumulative_unique_count(aware_repaired_keys)

aware_steps = [h["step"] for h in history_aware[:-1]]
dup_rejections_aware = np.cumsum([h["duplicate_rejections"] for h in history_aware[:-1]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    range(len(naive_unique_counts)),
    naive_unique_counts,
    "-o",
    linewidth=2.0,
    label="Naive latent BO",
)
axes[0].plot(
    range(len(aware_unique_counts)),
    aware_unique_counts,
    "-o",
    linewidth=2.0,
    label="Repair-aware BO",
)
axes[0].set_title("Cumulative unique repaired designs visited", fontsize=18, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Unique repaired designs", fontsize=18, fontweight="bold")
style_ax(axes[0])
axes[0].legend(prop={"size": 10, "weight": "bold"})

axes[1].plot(
    aware_steps,
    dup_rejections_aware,
    "-o",
    linewidth=2.0,
    label="Repair-aware BO",
)
axes[1].set_title("Cumulative duplicate repaired-design rejections", fontsize=18, fontweight="bold")
axes[1].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[1].set_ylabel("Cumulative rejections", fontsize=18, fontweight="bold")
style_ax(axes[1])
axes[1].legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 12. Summarising the best repaired catalyst design found by each BO method

After comparing the optimisation trajectories and the repaired-design diversity of the two BO loops, it is useful to finish with a direct summary of the **best repaired catalyst condition** found by each method.

That is exactly what this cell does.

Rather than looking at the full history, it extracts the final best point from each BO run and converts it into a user-facing experimental summary.

This gives a compact side-by-side view of the final optimisation outcome.

---

### What the code is doing

The helper function

```python
def best_design_summary(history, label):
```

takes a BO history together with a method label and then:

1. extracts the best raw latent point stored in that history,
2. extracts its best score,
3. decodes the raw point into experimental variables,
4. repairs it into a feasible catalyst setting,
5. and returns a dictionary summarising the resulting design.

So this function is effectively turning the final BO result into something that looks like an experimental recommendation.

The dataframe

```python
summary_df = pd.DataFrame([
    best_design_summary(history_naive, "Naive latent BO"),
    best_design_summary(history_aware, "Repair-aware BO"),
])
```

then places the two methods side by side for comparison.

---

### Why this summary is useful

The earlier figures showed the **path** taken by each BO method:

- how quickly the best score improved,
- how many unique repaired designs were visited,
- and how many duplicate repaired designs were rejected.

This cell instead focuses on the **final endpoint**.

That is important because, in an optimisation problem, it is not enough to know only how the search behaved.
We also want to know what experimental condition each method would ultimately recommend.

So this summary connects the abstract BO history back to the actual catalyst-design variables we care about.

---

### Why decoding and repair are applied again here

The history stores the best point in **latent** coordinates.

But the meaningful result is not the latent point itself.
It is the repaired feasible catalyst condition that would actually be run in practice.

So the code again performs:

> **best latent point → decode → repair**

before constructing the summary table.

This ensures that the final comparison is made in terms of:

- temperature,
- loading,
- time,
- solvent,
- and repair information,

rather than raw internal BO coordinates.

---

### What this result shows

A particularly important feature of the current result is that **both algorithms arrive at the same final optimised catalyst condition**.

In other words:

- the **naïve latent BO** loop
- and the **repair-aware BO** loop

ultimately recommend the same repaired feasible design, with the same best score.

That is an important conclusion and should be stated explicitly.

It means that, in this toy problem, the repair-aware BO loop does **not necessarily change the identity of the final optimum**.

Instead, its main advantage is that it reaches that strong repaired design **much earlier and more efficiently**.

So the central lesson is not:

> repair-aware BO always finds a different best solution

but rather:

> repair-aware BO can reach the same optimised condition with fewer wasted BO iterations and better repaired-design efficiency.

That is actually a very realistic and practically important outcome.

---

### Why this is not a contradiction

At first sight, one might wonder why the two methods can end at the same final optimised condition if one is clearly more informed than the other.

But this is perfectly sensible.

Both methods are still optimising the **same repaired experimental objective**.
So if the toy problem has one clearly best repaired catalyst condition, then both BO loops can eventually find it.

The difference lies in the **route** to that optimum:

- the naïve method wastes more steps revisiting equivalent repaired designs,
- whereas the repair-aware method reaches the same strong repaired region much faster.

So the final optimum can be the same even when the optimisation process is meaningfully different.

---

### Why the repair information still matters here

Even though the final repaired condition is the same, the repair flags can still differ between the two methods.

That is because the two algorithms may reach the same repaired design from **different latent raw points**.

So two different latent proposals can collapse onto the same repaired catalyst setting while carrying different repair histories.

This again reinforces one of the main conceptual lessons of the notebook:

> in a mixed-variable constrained BO problem, the raw latent point and the repaired experimental design are not the same object.

That is why the dataframe includes not only the final design variables, but also:

- whether repair occurred,
- and which repair flags were triggered.

---

### What this summary contributes to the notebook

This cell gives the notebook a clean and concrete way to conclude the comparison.

The earlier plots show that the repair-aware method is more efficient during optimisation.

This final table then shows that both methods ultimately converge to the same repaired catalyst condition.

So taken together, the message becomes:

- the final optimised condition may be the same,
- but the repair-aware BO loop gets there more efficiently.

That is a strong and realistic conclusion for a constrained mixed-variable BO tutorial.

---

### Key takeaway

This cell summarises the final best repaired catalyst design found by each BO method.

The key result is that **both algorithms arrive at the same optimised repaired condition**, even though their optimisation paths are different.

So in this notebook, the main advantage of repair-aware BO is not necessarily finding a different final optimum, but reaching the same strong repaired feasible design **faster and with fewer redundant repaired-design proposals**.

In [ ]:
def best_design_summary(history, label):
    best_x = history[-1]["best_x"]
    best_y = float(history[-1]["best_y"])
    decoded = decode_single_design(best_x)
    repaired = repair_single_design(decoded)

    return {
        "Method": label,
        "Best score": best_y,
        "Best repaired temperature": repaired["temperature"],
        "Best repaired loading": repaired["loading"],
        "Best repaired time": repaired["time"],
        "Best repaired solvent": repaired["solvent"],
        "Best design was repaired": repaired["was_repaired"],
        "Repair flags": ", ".join(repaired["repair_flags"]) if repaired["repair_flags"] else "None",
    }

summary_df = pd.DataFrame([
    best_design_summary(history_naive, "Naive latent BO"),
    best_design_summary(history_aware, "Repair-aware BO"),
])

summary_df

## 🧭 Closing Remarks

In this tutorial, we moved from **standard batch Bayesian Optimisation in a continuous design space** to the problem of running BO in a **mixed-variable, constrained experimental setting**.

The central idea was that once the design space contains:

- continuous variables,
- discrete variables,
- categorical choices,
- and explicit feasibility rules,

the BO problem changes in an important way.

The challenge is no longer only:

- how to fit a surrogate,
- how to define an acquisition function,
- and how to repeat the BO loop,

but also:

- how to represent a design space that is not purely continuous,
- how to convert raw BO proposals into actual experimental settings,
- how to enforce feasibility in a meaningful way,
- and how to decide whether two different latent proposals really correspond to two different experiments.

At a structural level, the BO workflow still followed the same general pattern:

1. fit a Gaussian Process surrogate to the currently observed data,
2. build an acquisition rule from that surrogate,
3. propose the next candidate,
4. evaluate the objective there,
5. update the dataset,
6. and repeat.

But the meaning of a candidate changed.

In the earlier notebooks, a BO proposal could be interpreted directly as a point in the search space.

Here, that was no longer possible.

A proposed latent point first had to be transformed through:

> **decode → repair → evaluate**

That shift was the whole point of the notebook.

Instead of treating BO as operating directly on the final design space, we treated it as something that must work when the actual experiment is only obtained **after** a proposal has been decoded and repaired into a feasible condition.

That is much closer to many real optimisation problems.

Across the notebook, we replaced abstract continuous benchmarks with a synthetic **catalyst optimisation** problem involving:

- temperature,
- loading,
- reaction time,
- and solvent identity,

together with solvent-dependent feasibility rules and realistic grid snapping.

That change was important, because it meant the optimisation problem now resembled a structured experimental design space rather than a simple continuous box.

What changed was not the BO loop itself, but the **relationship between latent proposals and actual experiments**.

The first important lesson was that mixed-variable constrained BO is not just continuous BO with a few post-processing steps added on top.

Once variables are mixed and constraints matter, a raw latent candidate is no longer the object we care about most.

The meaningful object is the **repaired feasible design**.

That is why this tutorial introduced:

- a latent continuous representation,
- explicit decoding functions,
- explicit repair rules,
- solvent-specific feasibility windows,
- and a repaired objective that evaluates only the final feasible experimental condition.

So one of the main conceptual takeaways was that:

> in mixed-variable constrained BO, the meaningful unit of optimisation is often the **repaired feasible experiment**, not the raw latent proposal.

The notebook then showed that this has immediate consequences for how BO behaviour should be interpreted.

Two different latent points may decode and repair to the **same final catalyst condition**.
So latent-space novelty does not necessarily imply experimental novelty.

That led to one of the most important practical lessons of the notebook:

> in a constrained mixed-variable setting, it can matter not only whether BO proposes a new latent point, but whether it proposes a **new repaired design**.

That distinction was central to the comparison between the two BO loops.

The **naïve latent BO** loop still optimised the same repaired objective, but it accepted candidates solely based on the standard latent-space BO logic.

The **repair-aware BO** loop changed only one thing, but it changed something important:

- it tracked which repaired feasible designs had already been visited,
- and it rejected new latent proposals if they would collapse onto repaired experiments that were already known.

This was the main methodological achievement of the notebook.

It showed that BO can be made more faithful to the actual experimental problem without replacing the whole BO framework.

That is exactly what made the repair-aware loop better.

It was better **not because it used a different surrogate**,
not because it used a different acquisition function,
and not because it changed the optimisation problem itself.

It was better because it reasoned in terms of the **actual feasible experiments** rather than only in terms of raw latent coordinates.

The diagnostics then made this difference concrete.

The best-score comparison showed that repair-aware BO could reach the strong repaired region **much earlier**.

The repaired-design diversity plots showed that:

- the repair-aware loop kept discovering new repaired designs steadily,
- while the naïve loop often plateaued because different latent proposals were collapsing onto designs that were not genuinely new.

The duplicate-rejection diagnostic then made the mechanism visible:

> a large number of acquisition-guided latent proposals were being blocked because they would have produced repaired experiments that had already been seen.

This was one of the clearest results in the notebook.

It showed that the repair-aware mechanism was not just a cosmetic modification.
It was actively changing the optimisation trajectory in a meaningful way.

At the same time, the notebook also made clear that the two methods could still converge to the **same final optimised catalyst condition**.

That is not a weakness of the result.
It is actually a realistic and important lesson.

The repair-aware loop was better **not necessarily because it found a different final optimum**, but because it reached the same strong repaired design **faster and more efficiently**.

That is often exactly what matters in practice.

If one BO iteration corresponds to:

- a synthesis cycle,
- a reaction campaign,
- an instrument run,
- or a round of simulation jobs,

then reaching the same optimum in fewer wasted iterations is a real advantage.

So one of the most important practical conclusions of the notebook was:

> better BO behaviour does not always mean a different final optimum; it can also mean a **more efficient path** to the same optimum.

That is where this notebook connects most directly to real research.

In a real experimental workflow, optimisation variables are often not purely continuous.
They may include:

- temperatures set in coarse increments,
- times chosen from predefined levels,
- solvents chosen categorically,
- and compatibility constraints that make many nominal combinations impossible or meaningless.

In such settings, the researcher is often not asking only:

> what is the next best latent point?

but rather:

> what is the next **runnable experiment**, and have I effectively already tried it in repaired form?

That is a genuinely mixed-variable constrained BO question.

So by the end of this notebook, we have not introduced an entirely different optimisation paradigm.
Instead, we have taken the BO workflow already built earlier and adapted it to a much more realistic kind of design space:

- one with mixed variable types,
- one with explicit constraints,
- one in which raw proposals must be decoded and repaired,
- and one in which repaired experimental novelty matters.

That gives us a natural stopping point:

> we now know how to build, run, and interpret a practical mixed-variable constrained Bayesian Optimisation workflow, and why a repair-aware BO loop can be better than a naïve latent-space loop.

The next stage will naturally ask what happens when these ideas are pushed even further:

- when constraints become more complex,
- when objectives become noisy or multi-objective,
- when batches and mixed variables must be handled together,
- or when BO must be integrated more directly into real human-driven experimental decision-making.

That is where mixed-variable constrained BO begins to move from a useful modelling idea to a genuinely practical optimisation tool.